In [ ]:
# === ARRANQUE EN COLAB: arbol de carpetas de la sesion =====================
# Este cuaderno se escribio para correr desde la carpeta `notebook/` de su
# sesion, con ../data, ../figuras y ../resultados al lado. Colab arranca en
# /content y sin ese arbol, asi que aqui se recrea y nos situamos dentro: con
# eso, todas las rutas relativas del cuaderno funcionan igual que en local.
import os, sys

if "google.colab" in sys.modules:
    _RAIZ = "/content/S04_regresion_multiple"
    for _sub in ("notebook", "data", "figuras", "resultados"):
        os.makedirs(os.path.join(_RAIZ, _sub), exist_ok=True)
    os.chdir(os.path.join(_RAIZ, "notebook"))
    print("Colab: carpeta de trabajo en", os.getcwd())



# Sesión 4 — Regresión Lineal Múltiple: diagnóstico, dummies e interacciones

**Curso:** Herramientas para la Ciencia de Datos, Facultad de Negocios, UPC
**Programa:** Administración y Ciencia de Datos para Negocios
**Laboratorio de replicación:** De Cock (2011), *Ames, Iowa: Alternative to the Boston Housing Data* — el dataset docente que reemplazó al *Boston Housing* — y caso de negocio de valoración de vivienda sobre el mismo *Ames Housing*.

> **Cómo se abre este cuaderno.** El curso lo distribuye por **Google Drive**: en la
> carpeta compartida, clic derecho sobre el archivo → *Abrir con* → *Google
> Colaboratory*. Conviene empezar por **Archivo → Guardar una copia en Drive** para
> conservar el trabajo. No se requiere cuenta de GitHub ni instalar nada en el equipo:
> los datos de la sesión viajan dentro del propio cuaderno.
> **Carpeta del curso en Drive (Pregrado):** https://drive.google.com/drive/folders/1-YJxRt0n-UZwQCu03Lls2LGUYz6KMsl2


---

## 1. Objetivos de aprendizaje

Al terminar la sesión, el estudiante:

1. Extiende el OLS a **múltiples predictores** e interpreta los **coeficientes parciales** («manteniendo lo demás constante»), distinguiéndolos de la correlación bruta.
2. Detecta y corrige **multicolinealidad** (VIF), **heterocedasticidad** (Breusch-Pagan → errores robustos White/HC3) y **autocorrelación** (Durbin-Watson); identifica observaciones **influyentes** (distancia de Cook).
3. Incorpora variables cualitativas (**dummies / one-hot**, sin dummy redundante) e **interacciones** (zona × tamaño), con lectura correcta de sus efectos condicionales.
4. Aplica **transformaciones** (log, Box-Cox) cuando la respuesta es asimétrica.
5. Compara modelos por **validación cruzada k-fold** para elegir el más generalizable.

## 2. Mapa de la sesión: nueve capítulos en dos clases

La sesión ocupa **dos clases**. Cada capítulo lleva un código —4.1 a 4.9— que es **el mismo** en el sílabo, en la guía del docente, en la guía de laboratorio y en las diapositivas, de modo que se pueda pasar de un material a otro sin traducir numeraciones.

**JUEVES — 145 min de contenido** (bloque A1 de 75, receso de 15, bloque A2 de 70; antes, 20 min de control sobre la Sesión 3)

| Cód. | Pregunta que responde | Dónde vive en este cuaderno |
|---|---|---|
| **4.1** | ¿Por qué una sola variable no explica el precio de una vivienda? | Sin celdas: se abre en clase con la tasación de una sola variable |
| **4.2** | ¿Qué es un coeficiente parcial? | «Teoría guiada» |
| **4.3** | ¿Cómo se obtiene el modelo con varios predictores? | «Teoría guiada» |
| **4.4** | ¿Cómo se mide si la incorporación de variables aportó valor? | «Teoría guiada» |
| **4.5** | ¿Se sostiene con datos reales? La réplica de Ames (De Cock, 2011) | «La réplica de Ames (De Cock, 2011)» — laboratorio, pasos 0 a 5 |

**VIERNES — 120 min corridos**

| Cód. | Pregunta que responde | Dónde vive en este cuaderno |
|---|---|---|
| **4.6** | ¿Cuándo se puede confiar en un modelo con múltiples variables? | «Supuestos: cómo identificarlos y corregirlos» |
| **4.7** | ¿Cómo se verifica que el desempeño es real? | «Validación cruzada k-fold» y «Panel de incertidumbre verificado» |
| **4.8** | ¿Qué decisión habilita? Tasación de vivienda | «Laboratorio de negocio: tasación de vivienda» — laboratorio, paso 6 |
| **4.9** | ¿Qué no se puede afirmar, y qué sigue en S05? | «Cierre» |

> El **control** de esta sesión se resuelve en aula, en la franja de 20 minutos del jueves siguiente, y cubre **los nueve capítulos**, de los dos días.


## Cómo leer este cuaderno

Este cuaderno no solo **se ejecuta**: enseña cada paso. Convenciones:

- **❓ Qué se quiere averiguar** abre cada resultado importante: la pregunta que ese resultado contesta, qué decisión depende de ella y **qué significaría cada resultado posible, dicho antes de ver el número**. Conviene detenerse ahí y contestar mentalmente antes de ejecutar: un dato solo informa a quien traía una pregunta.
- **🔎 Qué hace este código** precede a cada celda de código; **📖 Cómo se lee esta salida** sigue a cada resultado numérico clave. **💡** añade intuición y **⚠️** marca un supuesto o alerta.
- El alumno **desarrolla la mecánica de forma manual** y la verifica contra la librería con `assert`: **🖐️ Cálculo manual**, **🧮 Matemática en el cuerpo** (derivación en LaTeX), **✅ Verificación desde la base** (recomputa el resultado y lo cruza con el Excel) y **🧱 Construcción de la regresión múltiple desde cero**.
- **📄 En el paper** indica la procedencia exacta de cada resultado replicado (fuente, sección, página).
- La **«Sección 8» (Supuestos)** ejecuta los diagnósticos; su teoría vive en la guía de supuestos de la sesión.
- **Valor operativo vs. benchmark:** las cifras que se ejecutan aquí son las del **venv** (mandan); las del **paper** (De Cock) se citan como *benchmark* etiquetado. Pequeñas diferencias provienen del método/versión del dataset y caen dentro de tolerancia (±0,01).
- **Convención Excel:** los resultados del modelo se vuelcan a `resultados/S04_resultados.xlsx` y las **figuras de resultados se generan leyendo ese Excel**. Los bloques de descomposición y de supuestos **no escriben** en él.

Materiales hermanos: `laboratorio/GUIA_LABORATORIO_S04.docx`, `plantillas/checklist_diagnostico_regresion.docx`, `plantillas/comparacion_modelos_cv.docx`, `plantillas/diccionario_ames.docx`, `evaluacion/drills.docx`, `evaluacion/entregable.docx`, el cuaderno de la sesión, las fuentes de actualidad de la sesión y la fuente canónica de supuestos la guía de supuestos de la sesión.

## Preparación del entorno

La celda siguiente reconstruye, **solo cuando el cuaderno se ejecuta en Google Colab**, el **entorno certificado del curso**: instala con versión fijada los siete paquetes de la matriz de la matriz de versiones certificada del curso (los mismos cinco que el material de referencia de la sesión exige en su `VENV_REF`, más `matplotlib` y `seaborn`), reintenta **paquete por paquete** si algún tag no resuelve, e imprime al final las versiones **efectivamente instaladas**, con aviso de cualquier deriva. En una ejecución local con el venv del curso ya configurado, se omite automáticamente.

**🔎 Qué hace este código.** Instala, **solo en Google Colab**, los siete paquetes de la sesión con **versión fijada** (`numpy 2.5.1`, `pandas 2.3.3`, `scipy 1.16.3`, `matplotlib 3.11.1`, `scikit-learn 1.6.1`, `statsmodels 0.14.6`, `seaborn 0.13.2`) y luego **reporta las versiones realmente instaladas**. En ejecución local la celda se salta sola.

**Capa docente — por qué los pines son siete y por qué el reintento es *por paquete*.** El validador el material de referencia de la sesión compara el entorno contra `VENV_REF` (`numpy`, `pandas`, `scipy`, `scikit-learn`, `statsmodels`) y **aborta con `exit 2`** si alguna difiere: sin `pandas` ni `scipy` fijados, un Colab reciente daría «entorno no certificable» aunque el código sea correcto. `matplotlib` se fija porque las figuras del contrato se renderizan con él, y `seaborn` porque el mapa de calor depende de su API. El reintento se hace **paquete por paquete**: si un solo tag no resuelve en el runtime del día, únicamente **ese** paquete se degrada a la versión disponible y los demás conservan su pin. **Si el reintento fuera todo-o-nada** —un `except` que reinstala la lista entera sin fijar— un fallo aislado anularía los siete pines y el cuaderno correría con las versiones preinstaladas de Colab: deriva silenciosa en los últimos decimales, sin que nada avise. Por eso, además, la celda imprime la tabla «certificada → instalada» y marca `DERIVA` explícita.

In [ ]:
# SKIP-LOCAL: solo Colab.
# Colab ya trae el nucleo cientifico (numpy, pandas, scipy, matplotlib, seaborn,
# scikit-learn, statsmodels, openpyxl) COMPILADO ENTRE SI. Reinstalarlo con las
# versiones del venv del curso ROMPE el entorno: scipy y statsmodels dejan de
# importar con "cannot import name '_slice' from 'numpy._core.umath'". Por eso
# aqui solo se instala lo que Colab NO trae.
import sys

# Trazabilidad (sin reinstalar): versiones en uso frente a la matriz
# certificada del curso en la matriz de versiones certificada del curso. Si alguna difiere, las cifras
# pueden variar en los ultimos decimales; el metodo y las conclusiones no.
import importlib.metadata as _md

_CERTIFICADAS = {
    "matplotlib": "3.11.1",
    "numpy": "2.5.1",
    "pandas": "2.3.3",
    "scikit-learn": "1.6.1",
    "scipy": "1.16.3",
    "seaborn": "0.13.2",
    "statsmodels": "0.14.6",
}

print(f"{'paquete':18}{'en uso':14}{'certificada':14}estado")
for _p, _cert in _CERTIFICADAS.items():
    try:
        _v = _md.version(_p)
    except Exception:
        _v = "ausente"
    _estado = "=" if _v == _cert else "distinta (se respeta la de Colab)"
    print(f"{_p:18}{_v:14}{_cert:14}{_estado}")


**🔎 Qué hace este código.** Importa las librerías, **fija los hilos BLAS a 1 antes de importar `numpy`**, fija la semilla de la validación cruzada, define la paleta UPC y el ayudante `mostrar` (guarda cada figura como PNG y la muestra en el cuaderno). El backend `Agg` genera figuras sin ventana gráfica.

**Capa docente — el pineado BLAS (`OMP_NUM_THREADS=1`) y por qué va ANTES del import.** Detrás de cada OLS hay álgebra matricial (`XᵀX`, su inversa, `Xᵀy`) que `numpy` no ejecuta por sí mismo: la delega en una librería **BLAS** (OpenBLAS o MKL), que **reparte cada suma entre varios hilos** para ir más rápido. La suma en punto flotante **no es asociativa** —`(a+b)+c` puede diferir de `a+(b+c)` en el último bit—, así que dos equipos con distinto número de núcleos suman en distinto orden y devuelven un `R²` o un `SE` que difieren en los decimales finales. Fijar los hilos a 1 impone **un único orden de suma** y vuelve el resultado reproducible bit a bit. Va **antes** de `import numpy` porque la BLAS lee esas variables de entorno **una sola vez, al cargarse**: escritas después, el `setdefault` no tiene ningún efecto y el pineado queda decorativo. Es el mismo bloque que abre el material de referencia de la sesión, y por eso el validador puede exigir coincidencias de 1e-10 entre dos vías de cálculo. **Si no estuviera:** el cuaderno correría igual, pero las cifras derivarían de una máquina a otra y los `assert` del contrato (tolerancias de 1e-6) podrían fallar sin que hubiera error alguno en el modelo. `np.random.seed(42)` cubre otra fuente de variación —el barajado de la validación cruzada—, y `warnings.filterwarnings("ignore")` solo silencia avisos de versión: no cambia ninguna cifra.

In [ ]:
# Configuración, imports y ayudantes
import warnings
warnings.filterwarnings("ignore")

import os, sys
from pathlib import Path

# Determinismo entre maquinas: fijar los hilos BLAS a 1 ANTES de importar numpy.
# Por que ANTES: la BLAS (OpenBLAS/MKL) lee estas variables UNA SOLA VEZ, al cargarse con
# numpy; escritas despues no tienen efecto. Por que a 1: la suma en punto flotante no es
# asociativa, y una BLAS multihilo reordena las sumas de XtX segun el nº de nucleos -> el
# R2/SE cambia en los ultimos decimales de una maquina a otra. Mismo pineado que
# el material de referencia de la sesión. En este equipo es un no-op numerico (verificado bit-a-bit).
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v, "1")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")            # backend headless: las figuras se guardan como PNG
import matplotlib.pyplot as plt
from IPython.display import Image, display
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor, OLSInfluence
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
from scipy import stats
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression

EN_COLAB = "google.colab" in sys.modules

# Reproducibilidad de la validación cruzada
np.random.seed(42)

# Paleta del curso
UPC_ROJO, UPC_TINTA, UPC_GRIS, UPC_ROSA = "#C8102E", "#1F2A44", "#8A8D8F", "#C9A3AE"
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})

def mostrar(fig, ruta):
    "Guarda la figura como PNG (150 dpi) y la muestra en el cuaderno."
    fig.tight_layout()
    fig.savefig(ruta, dpi=150, bbox_inches="tight")
    plt.close(fig)
    display(Image(str(ruta)))

print("Versiones — pandas", pd.__version__, "| numpy", np.__version__, "| statsmodels", sm.__version__)
print("Librerias cargadas. statsmodels y scikit-learn listos para OLS múltiple.")

**🔎 Qué hace este código.** Localiza `data/AmesHousing.csv` en local (venv) o en Colab; si no está, lo **descarga de la fuente primaria JSE** (De Cock) verificando el **SHA256** de los bytes crudos y el esquema 2930×82 (mismo contrato que `data/descargar_datos.py`). Después fija las rutas de `resultados/` (Excel) y `figuras/`. Deja lista la variable `RUTA_XLSX`, que apunta al Excel de contrato.

In [ ]:
# --- Rutas robustas: funcionan en local (venv) y en Colab ---
_CAND_DATA = [
    "data/AmesHousing.csv",                                   # cwd = carpeta de la sesión
    "../data/AmesHousing.csv",                                # cwd = carpeta notebook/ (nbconvert)
    "Sesiones/S04_regresion_multiple/data/AmesHousing.csv",   # cwd = raíz del repo
    "AmesHousing.csv",                                        # Colab (archivo subido o ya descargado)
]
RUTA_DATA = next((c for c in _CAND_DATA if os.path.exists(c)), None)

if RUTA_DATA is None:
    # Descarga automática VERIFICADA (mismo contrato de integridad que data/descargar_datos.py):
    # fuente primaria JSE (AmesHousing.txt de De Cock, tab-separado) + SHA256 de los bytes crudos.
    import hashlib
    import io as _io
    import urllib.request

    _URL_JSE = "http://jse.amstat.org/v19n3/decock/AmesHousing.txt"
    _SHA256_JSE = "6cfe6cb525ba437de428653a1040e2aed7d696640bf75203786a6d7a0e67cfcc"
    print("No se encontró 'AmesHousing.csv': descargando la fuente primaria de JSE (De Cock)...")
    try:
        _raw = urllib.request.urlopen(_URL_JSE, timeout=120).read()
    except Exception as _e:  # sin red: instrucción manual (comportamiento previo)
        raise FileNotFoundError(
            "No se encontró 'AmesHousing.csv' y la descarga de JSE falló (%r). Ejecute "
            "'data/descargar_datos.py' o suba el archivo al directorio de trabajo." % (_e,)) from _e
    _sha = hashlib.sha256(_raw).hexdigest()
    if _sha != _SHA256_JSE:
        raise RuntimeError(
            "SHA256 del AmesHousing.txt descargado NO coincide con el certificado.\n"
            "  esperado: %s\n  obtenido: %s\n"
            "Fuente posiblemente alterada: usar 'data/descargar_datos.py' o subir el CSV a mano."
            % (_SHA256_JSE, _sha))
    _df_jse = pd.read_csv(_io.BytesIO(_raw), sep="\t")
    if _df_jse.shape != (2930, 82):
        raise RuntimeError("Esquema inesperado %s (esperado (2930, 82))." % (_df_jse.shape,))
    _df_jse.to_csv("AmesHousing.csv", index=False)
    RUTA_DATA = "AmesHousing.csv"
    print("Descarga verificada (SHA256 ok, 2930 x 82) -> AmesHousing.csv")

SESION_DIR = os.path.dirname(os.path.dirname(os.path.abspath(RUTA_DATA)))
DATA_DIR = os.path.join(SESION_DIR, "data")
RESULTS_DIR = os.path.join(SESION_DIR, "resultados")
FIG_DIR = os.path.join(SESION_DIR, "figuras")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
RUTA_XLSX = os.path.join(RESULTS_DIR, "S04_resultados.xlsx")

print("Datos      ->", RUTA_DATA)
print("Resultados ->", RUTA_XLSX)
print("Figuras    ->", FIG_DIR)

**🔎 Qué hace este código.** Carga la **base original de De Cock** (`AmesHousing.csv`, 2930 × 82, nombres de columna con espacios) y verifica sus dimensiones. Es el dataframe único de toda la sesión: la réplica y el laboratorio parten de él.

In [ ]:
# Carga de la base original verificada (De Cock 2011, Ames Housing 2930 x 82)
ames = pd.read_csv(RUTA_DATA)
print("Dimensiones:", ames.shape, "(esperado 2930 × 82)")
print("Columnas clave presentes:",
      all(c in ames.columns for c in
          ["SalePrice", "Gr Liv Area", "Overall Qual", "Garage Cars",
           "Total Bsmt SF", "Year Built", "Neighborhood", "Sale Condition"]))

## 4.2 a 4.4 — Teoría guiada: el coeficiente parcial, cómo se obtiene el modelo y cómo se mide su aporte (Sección 3 del cuaderno)

Cada bloque combina una idea, una **minidemostración** ejecutable y su **lectura de negocio**. El contenido deriva de el glosario de la sesión e la guía de interpretación de resultados.

### De una a varias X: el coeficiente parcial — capítulo 4.2 (subsección 3.1)

El OLS múltiple estima `Y = β₀ + β₁X₁ + … + βₚXₚ + ε`. Cada `βⱼ` es el **efecto parcial** de `Xⱼ`: el cambio esperado en `Y` ante +1 unidad de `Xⱼ` **manteniendo constantes** los demás predictores (*ceteris paribus*). No es la correlación bruta.

### 🧮 Matemática en el cuerpo — forma matricial del OLS múltiple y R² ajustado

Con `n` observaciones y `p` predictores, se apilan la **matriz de diseño** `X` (de tamaño `n × (p+1)`, con una primera columna de unos para el intercepto) y el vector respuesta `y`. El OLS minimiza `SSE = (y − Xβ)ᵀ(y − Xβ)`; derivando e igualando a cero se obtienen las **ecuaciones normales** `XᵀX β = Xᵀy`, cuya solución es

$$\hat{\beta} = (X^{\top}X)^{-1}X^{\top}y.$$

La bondad de ajuste se resume con `R² = 1 − SSE/SST`. Como el `R²` **nunca baja** al añadir predictores, para **comparar** modelos con distinto número de variables se usa el **R² ajustado**, que penaliza los predictores que no aportan:

$$R^{2}_{\text{aj}} = 1 - \frac{(1-R^{2})\,(n-1)}{\,n-p-1\,}.$$

La forma matricial `β = (XᵀX)⁻¹Xᵀy` es la que reconstruye la «Sección 7» desde el CSV crudo.

**❓ Qué se quiere averiguar.** ¿Cuánto vale realmente un pie² adicional de vivienda, es decir, qué tarifa por superficie escribiría un tasador en su regla de valoración?

- **Qué decide:** la tarifa con la que se valora cada casa de la cartera. Si esa tarifa está inflada, toda ampliación se sobrevalora y la inmobiliaria paga de más por metro construido.
- **Antes de mirar el resultado:** si el coeficiente **no se moviera** al añadir `Overall Qual`, tamaño y calidad serían independientes y la tarifa bruta ya sería la buena. Si **baja**, parte de lo que se cobraba como «tamaño» era en realidad calidad —las casas grandes suelen estar mejor construidas— y la tarifa bruta sobrestima. Si llegara a **cambiar de signo**, la lectura bruta afirmaría justo lo contrario de la lectura correcta.

**🔎 Qué hace este código.** Ajusta `SalePrice ~ Gr Liv Area` y luego `SalePrice ~ Gr Liv Area + Overall Qual`, y compara el coeficiente de la superficie: cómo cambia al **descontar** la calidad. Usa el set limpio (sin las 5 atípicas > 4000 pie²).

In [ ]:
# Demo: cómo cambia el coeficiente de la superficie al añadir la calidad
dem = ames[ames["Gr Liv Area"] <= 4000].dropna(subset=["SalePrice", "Gr Liv Area", "Overall Qual"])

m_bruto = sm.OLS(dem["SalePrice"], sm.add_constant(dem[["Gr Liv Area"]])).fit()
m_parcial = sm.OLS(dem["SalePrice"], sm.add_constant(dem[["Gr Liv Area", "Overall Qual"]])).fit()

print("Coef. Gr Liv Area SOLO           : %.1f USD/pie2" % m_bruto.params["Gr Liv Area"])
print("Coef. Gr Liv Area + Overall Qual : %.1f USD/pie2 (parcial)" % m_parcial.params["Gr Liv Area"])

**📖 Cómo se lee.** Al descontar la calidad, el valor marginal del pie² **baja**: parte de lo que parecía «efecto del tamaño» era en realidad que las casas grandes suelen ser de mejor calidad. El coeficiente parcial aísla la contribución **neta** de cada factor: es la base del *pricing* hedónico y de la valoración masiva (AVM). **⚠️** Un coeficiente parcial es **asociación condicionada** a las variables incluidas, **no** un efecto causal (la guía de supuestos de la sesión, Parte 1.2).

### 🖐️ Cálculo manual — β̂ = (XᵀX)⁻¹Xᵀy sobre un subconjunto pequeño

Se toma un **subconjunto pequeño** (80 casas) y se resuelven las ecuaciones normales en forma matricial `β = (XᵀX)⁻¹Xᵀy` **sin llamar al ajuste de la librería**. Luego se verifica con `assert` que el resultado coincide con `statsmodels` sobre el **mismo** subconjunto: OLS múltiple no es una caja negra, es álgebra lineal.

**🔎 Qué hace este código.** Arma la matriz de diseño `X = [1, Gr Liv Area, Overall Qual]` de 80 filas, resuelve `β = (XᵀX)⁻¹Xᵀy` con `numpy`, y comprueba con `assert` que reproduce los coeficientes de `statsmodels` ajustado sobre las mismas 80 filas.

In [ ]:
# OLS múltiple a mano sobre un subconjunto pequeño (80 casas), verificado vs statsmodels
sub = dem[["Gr Liv Area", "Overall Qual", "SalePrice"]].head(80).to_numpy(float)
Xsub = np.column_stack([np.ones(len(sub)), sub[:, 0], sub[:, 1]])   # [1, área, calidad]
ysub = sub[:, 2]

beta_mano = np.linalg.solve(Xsub.T @ Xsub, Xsub.T @ ysub)           # β = (XᵀX)⁻¹Xᵀy
modelo_sub = sm.OLS(ysub, Xsub).fit()

print("           %12s %12s" % ("a mano", "statsmodels"))
for nombre, bm, bl in zip(["intercepto", "Gr Liv Area", "Overall Qual"], beta_mano, modelo_sub.params):
    print("%-12s %12.4f %12.4f" % (nombre, bm, bl))

assert np.allclose(beta_mano, modelo_sub.params, atol=1e-6)
print("\nOK: β = (XᵀX)⁻¹Xᵀy a mano reproduce statsmodels en el subconjunto (error < 1e-6).")

**📖 Cómo se lee.** Los tres coeficientes reconstruidos de forma manual igualan a `statsmodels` hasta el error de máquina. La misma álgebra, escalada a las 2923 casas, produce el modelo múltiple del contrato — se rearma completa en la «Sección 7».

### Multicolinealidad y VIF — capítulo 4.6 (subsección 3.2)

Cuando dos predictores están muy correlacionados aportan información **redundante** e **inflan la varianza** de los coeficientes. El **VIF** mide cuántas veces se infla esa varianza. Reglas prácticas: **VIF > 5** = colinealidad alta (atención); **VIF > 10** = severa. La multicolinealidad **no compromete la predicción**, solo la **interpretación** individual de cada coeficiente. Es **la estrella de S04** (la guía de supuestos de la sesión, Parte 1.5).

### 🧮 Matemática en el cuerpo — el factor de inflación de la varianza (VIF)

Para el predictor `Xⱼ`, se regresa `Xⱼ` contra **todos los demás predictores** y se toma el `R²ⱼ` de esa regresión auxiliar. El VIF es

$$\mathrm{VIF}_j = \frac{1}{1 - R_j^{2}}.$$

Si `Xⱼ` no está correlacionado con los demás, `R²ⱼ = 0` y `VIF = 1` (sin inflación). Si `Xⱼ` es casi combinación lineal de los otros, `R²ⱼ → 1` y el `VIF → ∞`. La raíz `√VIFⱼ` es el factor por el que se **agranda el error estándar** de `βⱼ` frente al caso sin colinealidad. El bloque 🖐️ siguiente reconstruye esta fórmula para `Overall Qual` y la verifica contra `variance_inflation_factor`.

**❓ Qué se quiere averiguar.** Si dos columnas del expediente miden lo mismo —plazas de garaje y pie² de garaje—, ¿se puede afirmar cuánto paga el mercado por **cada una por separado**?

- **Qué decide:** si el informe puede sostener la frase «una plaza de garaje adicional vale X USD» o solo «el garaje, en conjunto, vale Y». La colinealidad **no compromete la predicción**: invalida la interpretación individual, que es lo que se cita en una tasación.
- **Antes de mirar el resultado:** con `VIF ≈ 1` los predictores no se pisan y cada coeficiente es citable. Con `VIF > 5` la colinealidad es alta y con `VIF > 10` es severa: el modelo predice igual de bien, pero el reparto del efecto entre las dos variables se vuelve arbitrario y ninguno de los dos coeficientes es ya una cifra defendible.

**🔎 Qué hace este código.** Ilustra la colinealidad **en vivo**: incluye juntas `Garage Cars` y `Garage Area` (miden lo mismo, la capacidad del garaje) y calcula el VIF de cada predictor. El VIF del intercepto (`const`) **no se interpreta**.

In [ ]:
# Demo: Garage Cars y Garage Area son casi la misma variable -> VIF sube
g = ames.dropna(subset=["Garage Cars", "Garage Area", "Gr Liv Area", "Overall Qual"])
Xg = sm.add_constant(g[["Garage Cars", "Garage Area", "Gr Liv Area", "Overall Qual"]])
vif_demo = pd.DataFrame({
    "predictor": Xg.columns,
    "VIF": [variance_inflation_factor(Xg.values, i) for i in range(Xg.shape[1])],
})
print(vif_demo.round(3).to_string(index=False))
print("\nNota: el VIF de 'const' (intercepto) NO se interpreta.")

**📖 Cómo se lee.** `Garage Cars` y `Garage Area` suben a **VIF ≈ 5** cada una: el modelo no puede separar sus efectos porque describen lo mismo. Ante VIF alto se elige **una** de las dos, se combinan o se transforma; no se usan ambas si se van a interpretar coeficientes. En cambio `Gr Liv Area` y `Overall Qual` quedan holgadamente bajo 5.

### 🖐️ Cálculo manual — VIF = 1/(1−R²ⱼ) para Overall Qual

Se reconstruye el VIF del predictor `Overall Qual` **desde su definición**: se regresa `Overall Qual` contra los otros cuatro predictores del modelo múltiple, se toma el `R²ⱼ` y se calcula `1/(1−R²ⱼ)`. Se verifica con `assert` que reproduce `variance_inflation_factor` y el valor del contrato (**2,4735**).

**🔎 Qué hace este código.** Sobre el set limpio, ajusta la regresión auxiliar `Overall Qual ~ (los otros 4 predictores)`, toma su `R²ⱼ`, calcula `VIF = 1/(1−R²ⱼ)` y lo compara con `variance_inflation_factor` y con el target 2,4735.

In [ ]:
# VIF a mano = 1/(1-R²_j), regresando Overall Qual contra los demás predictores
PREDS = ["Overall Qual", "Gr Liv Area", "Garage Cars", "Total Bsmt SF", "Year Built"]
base_vif = ames[ames["Gr Liv Area"] <= 4000].dropna(subset=PREDS + ["SalePrice"])

otros = [p for p in PREDS if p != "Overall Qual"]
r2_aux = sm.OLS(base_vif["Overall Qual"], sm.add_constant(base_vif[otros])).fit().rsquared
vif_mano = 1.0 / (1.0 - r2_aux)

Xv = sm.add_constant(base_vif[PREDS])
vif_lib = variance_inflation_factor(Xv.values, 1)   # índice 1 = Overall Qual

print("R²_j (Overall Qual ~ otros) = %.4f" % r2_aux)
print("VIF a mano = 1/(1-R²_j)     = %.4f" % vif_mano)
print("VIF variance_inflation_factor = %.4f  (contrato 2.4735)" % vif_lib)

assert abs(vif_mano - vif_lib) < 1e-6
assert abs(vif_mano - 2.4735) < 1e-3
print("\nOK: el VIF a mano reproduce la librería y el contrato (2.4735).")

**📖 Cómo se lee.** El VIF no es un número mágico de la librería: es `1/(1−R²ⱼ)`. Con `R²ⱼ ≈ 0,60`, `Overall Qual` tiene `VIF ≈ 2,47`, muy por debajo de 5 ⇒ sin multicolinealidad preocupante. **💡** La raíz `√2,47 ≈ 1,57` dice que su error estándar es ~57 % mayor de lo que sería sin ninguna colinealidad — una inflación modesta y tolerable.

### Observaciones influyentes: distancia de Cook — capítulo 4.6 (subsección 3.3)

Una observación es **influyente** si al removerla cambian los coeficientes. La **distancia de Cook** `Dᵢ` resume ese efecto sobre todos los β; reglas prácticas: `Dᵢ > 1` = claramente influyente; `Dᵢ > 4/n` = **candidatos a inspección** (no a borrado automático).

**🔎 Qué hace este código.** Ajusta `SalePrice ~ Gr Liv Area + Overall Qual`, calcula la distancia de Cook de cada observación, y cuenta cuántas superan el umbral sensible `4/n`. Reporta también el `Dᵢ` máximo. *(Este conteo —~185 candidatos sobre n = 2925— corresponde a este modelo **didáctico de 2 predictores en nivel** `SalePrice ~ Gr Liv Area + Overall Qual`; el modelo múltiple final en log, de 5 predictores y n = 2923, da ~192 candidatos en la Sección 8: son umbrales `4/n` aplicados sobre modelos distintos.)*

In [ ]:
# Distancia de Cook del modelo SalePrice ~ Gr Liv Area + Overall Qual
mc = sm.OLS(dem["SalePrice"], sm.add_constant(dem[["Gr Liv Area", "Overall Qual"]])).fit()
cook = OLSInfluence(mc).cooks_distance[0]
umbral = 4 / len(dem)
print("Umbral 4/n = %.5f" % umbral)
print("Observaciones con D_i > 4/n: %d de %d" % ((cook > umbral).sum(), len(dem)))
print("D_i máximo: %.4f" % cook.max())

**📖 Cómo se lee.** El umbral `4/n` es **sensible** y marca varios puntos (~5-7 % de los datos): sirve para **listar candidatos**, no para eliminar. Primero se investiga *por qué* son extremos (p. ej. una venta no normal). El borrado solo se justifica con razón sustantiva — como las **5 casas > 4000 pie²** que De Cock recomienda remover (Sección 4).

### Diagnóstico de supuestos: residuales, Q-Q, Durbin-Watson, Breusch-Pagan — capítulo 4.6 (subsección 3.4)

El OLS asume **linealidad**, **independencia**, **homocedasticidad** (varianza constante) y **normalidad** de los residuales. Se diagnostica con:
- **Residuales vs. ajustados:** nube sin patrón ⇒ OK; embudo ⇒ heterocedasticidad; curva ⇒ no linealidad.
- **Q-Q plot:** puntos sobre la diagonal ⇒ residuales normales.
- **Durbin-Watson:** ≈ 2 ⇒ sin autocorrelación de primer orden (solo se lee si hay orden).
- **Breusch-Pagan:** `H₀ = homocedasticidad`; **p < 0,05 ⇒ hay heterocedasticidad** ⇒ usar errores robustos.

**❓ Qué se quiere averiguar.** ¿El modelo se equivoca lo mismo en una casa de 100 000 USD que en una de 700 000, o el error crece con el precio? De la respuesta depende que los p-valores e intervalos del informe signifiquen lo que dicen.

- **Qué decide:** si la inferencia se reporta tal como sale del `summary()` o hay que recalcularla con errores robustos **antes** de declarar «significativo» un factor de precio.
- **Antes de mirar el resultado:** Breusch-Pagan contrasta `H₀ = varianza constante`. Con `p ≥ 0,05` no hay evidencia de heterocedasticidad y los errores estándar clásicos son válidos sin corrección. Con `p < 0,05` la dispersión del error crece con el precio: los errores clásicos **subestiman** la incertidumbre y un *driver* puede parecer fiable sin serlo. Los dos remedios llegan enseguida, errores robustos (3.5) y transformación log (3.6).

**🔎 Qué hace este código.** Ajusta el modelo en **nivel** `SalePrice ~ Gr Liv Area` (sin log), y ejecuta la prueba de **Breusch-Pagan** y el estadístico **Durbin-Watson** para diagnosticar heterocedasticidad y autocorrelación.

In [ ]:
# Diagnóstico del modelo en NIVEL: Breusch-Pagan + Durbin-Watson
ms = sm.OLS(dem["SalePrice"], sm.add_constant(dem[["Gr Liv Area"]])).fit()
bp = het_breuschpagan(ms.resid, ms.model.exog)   # (LM, p_LM, F, p_F)
print("Modelo en NIVEL  SalePrice ~ Gr Liv Area")
print("  Breusch-Pagan LM = %.1f, p = %.2e  -> %s" % (
    bp[0], bp[1], "heterocedasticidad" if bp[1] < 0.05 else "homocedástico"))
print("  Durbin-Watson    = %.3f" % durbin_watson(ms.resid))

**📖 Cómo se lee.** El modelo en nivel muestra **heterocedasticidad severa** (Breusch-Pagan `p ≈ 0`): la dispersión del error crece con el precio. Eso invalida los errores estándar clásicos y puede hacer parecer «significativo» un factor que no lo es. Dos remedios que se ven en 3.5 y 3.6: **errores robustos** y **transformación log**.

### Errores estándar robustos (White / HC3) — capítulo 4.6 (subsección 3.5)

Con heterocedasticidad, los coeficientes OLS siguen **insesgados** pero sus **errores estándar** están mal. Los errores robustos (estimador de White; se usa **HC3** en muestras finitas) corrigen la matriz de covarianzas **sin cambiar los β**.

### 🧮 Matemática en el cuerpo — el error estándar robusto (HC3)

El OLS clásico estima `Var(β̂) = σ²(XᵀX)⁻¹`, que asume varianza constante. El estimador robusto de White reemplaza esa expresión por un **«sándwich»** que usa los residuales al cuadrado observados. La variante **HC3** (recomendada en muestra finita) pondera cada residual `eᵢ` por `(1 − hᵢ)²`, donde `hᵢ` es el *leverage* (elemento diagonal de la matriz sombrero):

$$\widehat{\operatorname{Var}}_{\text{HC3}}(\hat\beta) = (X^{\top}X)^{-1}\left(\sum_{i} \frac{e_i^{2}}{(1-h_i)^{2}}\, x_i x_i^{\top}\right)(X^{\top}X)^{-1}.$$

Los **β no cambian**; cambian sus errores estándar, y con ellos las `t` y los p-valores (se vuelven más conservadores). El bloque 🧱 de la «Sección 7» reconstruye esta matriz sándwich de forma manual.

**🔎 Qué hace este código.** Ajusta `SalePrice ~ Gr Liv Area` en nivel dos veces —con errores clásicos y con `cov_type="HC3"`— y compara el error estándar del coeficiente de la superficie. El coeficiente es idéntico; solo cambia su SE.

In [ ]:
# Errores estándar: clásico vs robusto (HC3), modelo en nivel
ols = sm.OLS(dem["SalePrice"], sm.add_constant(dem[["Gr Liv Area"]])).fit()
hc3 = sm.OLS(dem["SalePrice"], sm.add_constant(dem[["Gr Liv Area"]])).fit(cov_type="HC3")
print("Coeficiente Gr Liv Area (idéntico): %.2f" % ols.params["Gr Liv Area"])
print("  SE clásico      : %.3f" % ols.bse["Gr Liv Area"])
print("  SE robusto (HC3): %.3f" % hc3.bse["Gr Liv Area"])

**📖 Cómo se lee.** El SE robusto (~3,01) es **mayor** que el clásico (~2,08): el error clásico **subestimaba** la incertidumbre en ~45 %. Con heterocedasticidad confirmada, la significancia se juzga con los SE robustos — así se evita declarar un *driver* de precio como fiable cuando no lo es.

### Transformaciones: log y Box-Cox — capítulo 4.6 (subsección 3.6)

Cuando `Y` es asimétrica a la derecha (típico en precios), **log(Y)** estabiliza la varianza, aproxima la normalidad y cambia la interpretación a **cambios porcentuales** (≈ 100·β % por unidad de X). **Box-Cox** generaliza buscando el exponente `λ` óptimo (`λ = 0` equivale al log).

**🔎 Qué hace este código.** Mide la **asimetría** de `SalePrice` y de `log(SalePrice)`, y estima el `λ` de **Box-Cox**. Un `λ` cercano a 0 confirma que el log es la transformación adecuada.

In [ ]:
# Asimetría antes/después del log y lambda de Box-Cox
precio = dem["SalePrice"].values
print("Asimetría de SalePrice      : %.3f" % stats.skew(precio))
print("Asimetría de log(SalePrice) : %.3f" % stats.skew(np.log(precio)))
lmbda = stats.boxcox(precio)[1]
print("λ óptimo de Box-Cox         : %.3f  (≈ 0 confirma que el log es apropiado)" % lmbda)

**📖 Cómo se lee.** La asimetría cae de ~1,6 a ~0 con el log, y el `λ` de Box-Cox se acerca a 0: el `log(precio)` es el estándar de los modelos hedónicos porque convierte los coeficientes en **variaciones porcentuales**, más intuitivas («+1 punto de calidad ⇒ +x % de precio»).

### Variables cualitativas: dummy / one-hot y la dummy redundante — capítulo 4.2 (subsección 3.7)

Una categórica se codifica en columnas 0/1 (una por categoría). La **dummy redundante** (*dummy variable trap*) aparece al incluir *todas* las categorías **más** el intercepto: colinealidad **perfecta** (violación de MLR.3). Se evita al omitir una categoría de referencia (`drop_first=True`).

**🔎 Qué hace este código.** Genera las dummies de `Bldg Type` con `drop_first=True` y muestra cuántas columnas produce: una menos que el número de categorías (se omite la de referencia).

In [ ]:
# Dummies de 'Bldg Type' con categoría de referencia (drop_first=True)
d = pd.get_dummies(ames["Bldg Type"], prefix="Bldg", drop_first=True, dtype=int)
print("Categorías de 'Bldg Type':", ames["Bldg Type"].nunique(),
      "-> columnas dummy generadas:", d.shape[1], "(se omite la de referencia)")
print(d.head(3).to_string(index=False))

**📖 Cómo se lee.** Con `drop_first=True` cada coeficiente dummy se lee como **diferencia respecto a la categoría base** (p. ej. cuánto más caro es un tipo de vivienda frente al de referencia). Olvidar `drop_first` produce colinealidad perfecta: el modelo no estima. La demostración del error se ejecuta en la «Sección 8» (diagnóstico 3.1).

### Interacciones (efecto condicional) — capítulo 4.2 (subsección 3.8)

Un término `X₁·X₂` permite que el efecto de `X₁` **dependa** del valor de `X₂`. Principio de **jerarquía**: si se incluye la interacción, se incluyen sus términos principales.

### 🧮 Matemática en el cuerpo — el término de interacción y el efecto marginal

Con una variable continua `X₁` (superficie) y una dummy `D` (barrio premium), el modelo con interacción es

$$Y = \beta_0 + \beta_1 X_1 + \beta_2 D + \beta_3 (X_1\cdot D) + \varepsilon.$$

El **efecto marginal** de `X₁` sobre `Y` ya no es constante: es la derivada parcial

$$\frac{\partial\, E[Y\mid X]}{\partial X_1} = \beta_1 + \beta_3\, D,$$

es decir, `β₁` en el grupo de referencia (`D = 0`) y `β₁ + β₃` en el grupo premium (`D = 1`). El coeficiente `β₃` es **cuánto cambia la pendiente** del tamaño al pasar a barrio premium. Por eso `β₁` **no** se interpreta aislada como «el efecto del tamaño» cuando hay interacción.

**🔎 Qué hace este código.** Construye la dummy `premium` (barrios de alta gama) y la interacción `Gr Liv Area × premium`, ajusta el modelo respetando la jerarquía, e imprime la pendiente base, el extra de la interacción y la pendiente resultante en premium.

In [ ]:
# Demo mínima: el valor del pie2 cambia entre barrios premium y el resto
ipd = dem.copy()
ipd["premium"] = ipd["Neighborhood"].isin(["NridgHt", "NoRidge", "StoneBr"]).astype(int)
ipd["Area_x_premium"] = ipd["Gr Liv Area"] * ipd["premium"]
mi = sm.OLS(ipd["SalePrice"], sm.add_constant(
    ipd[["Gr Liv Area", "premium", "Area_x_premium"]])).fit()
print("Pendiente base (no premium): %.1f USD/pie2  (β1)" % mi.params["Gr Liv Area"])
print("Extra en premium           : %+.1f USD/pie2  (β3)" % mi.params["Area_x_premium"])
print("Pendiente en premium       : %.1f USD/pie2  (β1 + β3)" %
      (mi.params["Gr Liv Area"] + mi.params["Area_x_premium"]))

**📖 Cómo se lee.** El valor del metro cuadrado **no es igual en todos los barrios**: en zona premium cada pie² adicional vale bastante más (`β₁ + β₃` > `β₁`). Un modelo puramente aditivo no captura esto; la interacción sí. Es el núcleo del laboratorio (Sección 5).

### Validación cruzada k-fold — capítulo 4.7 (subsección 3.9)

Estima el desempeño **fuera de muestra**: se parte el dataset en *k* bloques, se entrena con *k−1* y se evalúa en el restante, rotando *k* veces, y se reporta la métrica **media ± desviación estándar**. Con `k=5` o `10` es el estándar; es la prueba honesta de generalización (mejor que el R² in-sample).

**🔎 Qué hace este código.** Ejecuta una CV 5-fold (semilla 42) del modelo `SalePrice ~ Gr Liv Area + Overall Qual` y reporta el `R²` medio y su desviación estándar entre folds.

In [ ]:
# Validación cruzada 5-fold del modelo de dos predictores
kf_demo = KFold(n_splits=5, shuffle=True, random_state=42)
Xcv = dem[["Gr Liv Area", "Overall Qual"]].values
ycv = dem["SalePrice"].values
r2_cv = cross_val_score(LinearRegression(), Xcv, ycv, cv=kf_demo, scoring="r2")
print("CV 5-fold R² por fold:", np.round(r2_cv, 4))
print("R² medio = %.4f  ±  %.4f (sd)" % (r2_cv.mean(), r2_cv.std()))

**📖 Cómo se lee.** El R² medio de CV mide qué tan bien predecirá el modelo en datos **nuevos**; la sd mide su estabilidad entre particiones. Un modelo con mayor R² in-sample puede generalizar peor: cuando el R² ajustado y la CV difieran, **prevalece la CV**.

## 4.5 — ¿Se sostiene con datos reales? La réplica de Ames (De Cock, 2011)

**Contexto.** El clásico *Boston Housing* fue retirado de scikit-learn por una variable con sesgo racial. De Cock publicó *Ames Housing* (2930 ventas residenciales en Ames, Iowa, 2006–2010; 80+ variables) como **alternativa docente** para regresión. Su ejemplo canónico es `SalePrice ~ Gr Liv Area` con **remoción de ventas atípicas**.

**Targets a reproducir** (declarado para la sesión de réplica del paper, tolerancia ±0,01 salvo el conteo):

| # | Resultado | Valor esperado |
|---|---|---|
| 1 | Correlación de Pearson `SalePrice ~ Gr Liv Area` (n=2930) | **0,7068** |
| 2 | Nº de ventas atípicas (`Gr Liv Area > 4000`) | **5** |
| 3 | R² simple `SalePrice ~ Gr Liv Area` tras remover las 5 (n=2925) | **0,518** |
| 4 | R² múltiple `log(SalePrice) ~ 5 predictores` (n=2923) | **0,847** |

*(Sin `assert` en esta sección: la validación numérica con tolerancias vive en el material de referencia de la sesión, que **recomputa desde la base**; la verificación cruzada con el Excel está en «Verificación desde la base».)*

### Sección 0 — Qué preguntaba De Cock, y por qué usó lo que usó (subsección 4.0)

**💡 Antes de tocar los datos.** Una réplica sin esta pregunta se vuelve mecánica: se ejecutan celdas y salen números. Lo que sigue explica **qué buscaba el autor** y **por qué el archivo que se acaba de cargar tiene la forma que tiene**, que es de donde proviene el criterio para diseñar un conjunto de datos propio mañana. Matiz de esta sesión: De Cock no propuso una técnica nueva, sino que **construyó y justificó un conjunto de datos docente**, de modo que sus decisiones de método son decisiones de **diseño de datos**. *(Desarrollo completo con las citas del original: la ficha de la sesión de réplica del paper, «Sección 0».)*

**El objetivo no era proponer un modelo** (p. 2). De Cock no modeló el mercado inmobiliario de Ames ni publicó un modelo de referencia: se negó a hacerlo, «to foil my more motivated students» (p. 13). Lo que declara buscar es distinto: «For a regression project, I was looking for a data set that would allow students the opportunity to display the skills they had learned within the class. The ideal data set needed to have a reasonably large number of variables and observations so that students would have to go beyond a simple algorithm, such as forward or stepwise selection, to construct a final model». En corto: **¿existe un conjunto real, actual y comprensible por un lego, con suficientes variables y observaciones para que ningún algoritmo automático de selección agote el problema?** El Boston Housing ya no servía —«the housing prices have become unrealistic for today's market»— y la alternativa de menor costo, un factor de actualización sobre los precios, la descartó porque «that would change the data from real to realistic».

**Con qué contaba** (pp. 2-3, 10). Los repositorios docentes de la época ofrecían candidatos, pero «the data sets were rather limited in the number of observations (n ≤ 100)»: la restricción de partida fue de **disponibilidad**, no de teoría ni de cómputo. La materia prima llegó de la Oficina del Asesor Municipal de Ames y llegó en bruto: «The initial Excel file contained 113 variables describing 3970 property sales that had occurred in Ames, Iowa between 2006 and 2010». Su ventaja decisiva es que **la selección de variables ya la hizo un profesional de la tasación**: «one would expect the assessor's office to only collect relevant information».

**Las cuatro decisiones de diseño que explican el dataframe cargado arriba.** (a) **De 113 a 80 variables**, con criterio de comprensibilidad —«a "layman's" data set that could be easily understood by users at all levels»—; lo retirado fueron sobre todo las «variables … related to weighting and adjustment factors used in the city's current modeling system» (p. 2), las piezas internas del modelo de tasación vigente. (b) **De 3970 a 2930 observaciones**, con dos filtros: las ventas no residenciales generaban «unusual conditions (observations with no living space or lot size)» —ceros estructurales que invalidarían un modelo de áreas (pp. 3-4)— y del centenar de viviendas vendidas más de una vez se conservó solo la más reciente, porque repetirlas «gave a greater weight to these particular homes» (p. 4). (c) **Las categóricas se entregan sin codificar** a propósito: «I give the students the data "as is"» (p. 3) — la codificación es parte del ejercicio. (d) **El tamaño obedece a una regla explícita**: «that the number of observations in the training set be six to ten times the number of variables» (p. 6); con 80 variables candidatas, ninguna base de n ≤ 100 podía cumplirla.

**Por qué la base llega SIN depurar** (p. 4). Aquí está la decisión que ordena todo lo que sigue: «no observations have been removed due to unusual values and all final residential sales from the initial data set are included in the data presented with this article». La regla de las viviendas de más de 4000 pies² es una **recomendación al docente, no una limpieza ya aplicada**, y su criterio de fondo no es el tamaño sino la **naturaleza de la transacción**: la mayoría de esas ventas son *Partial Sales* —obra incompleta al momento de la tasación— cuyo precio «likely don't represent actual market values»; las restantes son casas de gran superficie con precio coherente. El umbral en pies² es un **proxy operativo** y el diagnóstico prescrito es un gráfico: un diagrama de dispersión de precio contra superficie «will quickly indicate these points». Dos consejos más del texto anticipan celdas posteriores: la varianza no homogénea ya estaba diagnosticada en 2011 —«there is increasing variation with increasing price within the Ames housing market»— con su remedio por raíz cuadrada del precio (p. 9), y el barrio se recomienda como dummy porque así «the coefficients for the continuous variables tend to have values with more realistic interpretations» (p. 11).

**⚠️ De ahí una diferencia con el paper que conviene anticipar.** El artículo publica una regresión de cinco predictores con `R-Sq = 71,4 %` (Figura 5, p. 10) y **no es la que se replica aquí**: ese ajuste se estima sobre una submuestra reducida para cursos introductorios —solo ventas normales, viviendas por debajo de 1500 pies², unas doscientas observaciones al azar (p. 9)— cuyo tamaño exacto, semilla y composición **el paper no documenta**, de modo que ninguna réplica puede recuperarla fila a fila. Los cuatro targets de abajo se anclan a la base completa y a reanálisis propio, porque el autor decidió no publicar sus modelos. Lo que se juzga es la **conclusión** —una variable no basta, cinco bien elegidas explican la mayor parte del precio—, no un decimal.


**🔎 Qué hace este código.** Calcula el **Target #1**: la correlación de Pearson `SalePrice ~ Gr Liv Area` sobre el dataset completo (2930 casas).

In [ ]:
# Target #1 — Correlación de Pearson SalePrice ~ Gr Liv Area (dataset completo)
corr_saleprice_grlivarea = ames["SalePrice"].corr(ames["Gr Liv Area"])
print("Target #1  correlación = %.4f  (esperado 0.7068)" % corr_saleprice_grlivarea)

**📖 Cómo se lee.** La correlación ≈ **0,71** confirma la relación positiva fuerte que De Cock destaca con su scatterplot: la superficie habitable es el *driver* más visible del precio, aunque no el único.

**❓ Qué se quiere averiguar.** ¿Qué justifica borrar una venta de la base, y en qué se diferencia eso de quitar los datos que estorban?

- **Qué decide:** si se adopta la limpieza que recomienda De Cock y con qué argumento se defiende ante quien audite el modelo. Un filtro sin causa documentada es maquillaje; con causa, es control de calidad.
- **Antes de mirar el resultado:** si las 5 casas resultaran ventas `Normal` a precio de mercado, quitarlas sería recortar la muestra por conveniencia y habría que conservarlas. Si predominan las `Partial` —obra sin terminar, cuyo precio no es de mercado—, la remoción tiene causa de dominio y el umbral de 4000 pie² es solo el atajo operativo que las localiza.

**🔎 Qué hace este código.** Calcula el **Target #2**: filtra las ventas con `Gr Liv Area > 4000` pie² y muestra su `Sale Condition` para *entender* por qué se remueven.

In [ ]:
# Target #2 — Ventas atípicas: Gr Liv Area > 4000 pie2
atipicas = ames[ames["Gr Liv Area"] > 4000]
n_atipicas_gt4000 = len(atipicas)
print("Target #2  nº de atípicas = %d  (esperado 5)\n" % n_atipicas_gt4000)
print(atipicas[["Gr Liv Area", "SalePrice", "Sale Condition"]]
      .sort_values("Gr Liv Area", ascending=False).to_string(index=False))

**📖 Cómo se lee.** Son exactamente **5** casas. De las 5, **3 son ventas `Partial`** (obra sin terminar, precio no de mercado = los «true outliers») y **2 son mansiones** (`Abnorml` / `Normal`) vendidas a precio coherente pero de superficie excepcional. Se entiende la causa **antes** de filtrar: es remoción justificada por el dominio, no borrado ciego.

### 📄 En el paper — De Cock (2011) y la remoción de las 5 casas > 4000 pie²

- **De Cock, D. (2011).** *Ames, Iowa: Alternative to the Boston Housing Data as an End of Semester Regression Project.* **Journal of Statistics Education, 19(3).** DOI 10.1080/10691898.2011.11889627. PDF de acceso libre: https://jse.amstat.org/v19n3/decock.pdf
- La documentación oficial del dataset (`DataDocumentation.txt`) lo dice textualmente: *«There are 5 observations that an instructor may wish to remove … Three of them are true outliers (Partial Sales …) and two of them are simply unusual sales (very large houses priced relatively appropriately). I would recommend removing any houses with more than 4000 square feet …»*. La inspección confirma **3 `Partial` + 1 `Abnorml` + 1 `Normal`**.

Procedencia completa en la ficha de la sesión de réplica del paper (Sección 3). El valor **operativo** prevalece (venv); las cifras de De Cock son el *benchmark* etiquetado.

**🔎 Qué hace este código.** Aplica la limpieza de De Cock (`Gr Liv Area <= 4000`) y recalcula la correlación sobre el set limpio (2925 casas): sube al quitar las atípicas que la deprimían.

In [ ]:
# Paso 4 — Limpiar: remover las 5 (criterio Gr Liv Area <= 4000)
clean = ames[ames["Gr Liv Area"] <= 4000].copy()
print("Filas tras la limpieza:", len(clean), "(esperado 2925)")
print("Correlación tras remover las 5: %.4f (sube: las atípicas la deprimían)"
      % clean["SalePrice"].corr(clean["Gr Liv Area"]))

**📖 Cómo se lee.** La correlación sube de 0,7068 a ~0,72: unas pocas ventas anómalas **deprimían** la relación. Es el pago inmediato de investigar y remover con criterio.

**🔎 Qué hace este código.** Calcula el **Target #3**: ajusta el OLS simple `SalePrice ~ Gr Liv Area` sobre el set limpio y reporta su `R²` y su pendiente en USD por pie².

In [ ]:
# Target #3 — R² del modelo simple tras la limpieza
Xs = sm.add_constant(clean[["Gr Liv Area"]])
modelo_simple = sm.OLS(clean["SalePrice"], Xs).fit()
r2_simple_limpio = modelo_simple.rsquared
print("Target #3  R² simple = %.4f  (esperado 0.518)" % r2_simple_limpio)
print("Pendiente: %.1f USD por pie2 adicional" % modelo_simple.params["Gr Liv Area"])

**📖 Cómo se lee.** El modelo simple explica ~**52 %** de la varianza del precio (R² ≈ 0,518); la pendiente ≈ 116 USD/pie² es una primera regla de valoración. La superficie sola explica la mitad — la otra mitad la aportan calidad, garaje, sótano y antigüedad (Target #4).

**❓ Qué se quiere averiguar.** ¿Cuánto del precio de una vivienda queda explicado por cinco rasgos que un tasador anota en una visita: calidad, superficie, garaje, sótano y año?

- **Qué decide:** si una tasación automática con cinco campos es defendible o si el negocio necesita mantener la inspección completa. Es la cifra central de la réplica de De Cock.
- **Antes de mirar el resultado:** la superficie sola ya explicaba `R² ≈ 0,518` (Target #3). Si el modelo de cinco predictores se quedara cerca de ese 0,52, las cuatro variables añadidas no pagarían su costo de recolección. El *benchmark* etiquetado es **0,847** con tolerancia ±0,01, esto es, cerca del **85 %** de la variación del precio capturada con cinco campos.

**🔎 Qué hace este código.** Calcula el **Target #4**: pasa a `log(SalePrice)`, ajusta el modelo múltiple con los 5 predictores y reporta `R²`, `R²` ajustado, `F` global y su p-valor. `n = 2923` tras eliminar filas con faltantes.

In [ ]:
# Target #4 — Modelo múltiple sobre log(SalePrice) con 5 predictores
mult = clean.dropna(subset=PREDS + ["SalePrice"]).copy()
print("Observaciones del modelo múltiple:", len(mult), "(esperado 2923)")

y_log = np.log(mult["SalePrice"])
Xm = sm.add_constant(mult[PREDS])
modelo_multiple = sm.OLS(y_log, Xm).fit()
r2_multiple_log = modelo_multiple.rsquared
print("Target #4  R² múltiple (log) = %.4f  (esperado 0.847)" % r2_multiple_log)
print("R² ajustado = %.4f | F global = %.1f | p(F) = %.2e"
      % (modelo_multiple.rsquared_adj, modelo_multiple.fvalue, modelo_multiple.f_pvalue))

**📖 Cómo se lee.** Pasar de 1 a 5 predictores lleva el `R²` de 0,52 a ~**0,85**: casi **duplica** el poder explicativo. El `R²` ajustado casi iguala al `R²` (con n grande y p pequeño apenas penaliza) y la `F` global es muy elevada (`p ≈ 0`): el conjunto de predictores es altamente significativo.

**🔎 Qué hace este código.** Arma la tabla de coeficientes del modelo múltiple (log) con su SE y p-valor, y añade la columna `efecto_%_aprox = (e^β − 1)·100`: la lectura **porcentual** propia de un modelo en log. Esta tabla se vuelca al Excel en la hoja `coeficientes_multiple` (Sección 6), la fuente única de la que la guía y el entregable citan los efectos porcentuales (por ejemplo, +10,9 % por punto de calidad y +6,2 % por plaza de garaje). Además, como el modelo en log **rechaza la homocedasticidad** (Breusch-Pagan LM≈83, p≈0; ver Sección 8), junto a los SE clásicos se calculan los **errores estándar robustos HC3** con sus `t`, `p` e **intervalos de confianza al 95 %**: los coeficientes puntuales no cambian, solo su inferencia (el SE honesto bajo heterocedasticidad). Esta inferencia robusta se guarda en la hoja nueva `inferencia_robusta` del Excel (Sección 6).

In [ ]:
# Tabla de coeficientes del modelo múltiple (log): lectura porcentual
coef = pd.DataFrame({
    "coef": modelo_multiple.params,
    "SE": modelo_multiple.bse,
    "p_valor": modelo_multiple.pvalues,
})
coef["efecto_%_aprox"] = (np.exp(coef["coef"]) - 1) * 100
print(coef.round(5).to_string())

# Inferencia robusta HC3 (el modelo log rechaza homocedasticidad: BP LM≈83)
modelo_multiple_hc3 = sm.OLS(y_log, Xm).fit(cov_type="HC3")
ic_hc3 = modelo_multiple_hc3.conf_int()
robusta = pd.DataFrame({
    "coef": modelo_multiple_hc3.params,
    "SE_HC3": modelo_multiple_hc3.bse,
    "t_HC3": modelo_multiple_hc3.tvalues,
    "p_HC3": modelo_multiple_hc3.pvalues,
    "IC95_bajo": ic_hc3[0],
    "IC95_alto": ic_hc3[1],
})
print("\nInferencia robusta HC3 (modelo log) — el coef no cambia, solo su SE/t/p/IC:")
print(robusta.round(6).to_string())

**📖 Comparación con De Cock e interpretación.**
- Los cuatro targets caen dentro de tolerancia: **corr ≈ 0,71**, **5 atípicas exactas**, **R² simple ≈ 0,52**, **R² múltiple (log) ≈ 0,85**.
- Con `log(precio)`, cada coeficiente se lee en **porcentaje**: +1 punto de `Overall Qual` ≈ **+11 %** de precio; +1 plaza de garaje ≈ **+6 %**; +1 año más nuevo ≈ **+0,25 %**; cada 100 pie² ≈ **+2,8 %**.
- Los `R²` de statsmodels no serán idénticos a los de R/S-Plus del paper (diferencias numéricas y de versión del dataset); por eso la tolerancia ±0,01. Se reproduce el **orden de magnitud y la conclusión**, que es el objetivo docente.

## 4.8 — ¿Qué decisión habilita? Laboratorio de negocio: tasación de vivienda con dummies, interacción y elección por CV (Sección 5 del cuaderno)

**Dataset usado:** `AmesHousing.csv` completo de De Cock (2930 filas, ya limpio a 2925 tras remover las 5 atípicas). El conjunto completo contiene los tres barrios premium (`NridgHt`, `NoRidge`, `StoneBr`) con suficientes casas para estimar la interacción.

**Objetivo:** construir un modelo de precios con **≥1 dummy** (zona premium) y **1 interacción** zona × tamaño, diagnosticarlo (VIF, Breusch-Pagan → robustos), elegir entre dos modelos por **CV 5-fold** e interpretar los coeficientes en unidades de negocio.

Sigue paso a paso la `laboratorio/GUIA_LABORATORIO_S04.docx` y apóyate en `plantillas/checklist_diagnostico_regresion.docx` y `plantillas/comparacion_modelos_cv.docx`.

**🔎 Qué hace este código.** Construye la dummy `premium` (categoría base = «no premium», sin dummy redundante) y la interacción `Gr Liv Area × premium` sobre el set limpio con los 7 campos sin faltantes.

In [ ]:
# Dummy de zona premium (evita la trampa de la dummy: categoría base = "no premium")
BARRIOS_PREMIUM = ["NridgHt", "NoRidge", "StoneBr"]
lab = clean.dropna(subset=["SalePrice", "Gr Liv Area", "Neighborhood",
                           "Overall Qual", "Garage Cars", "Total Bsmt SF",
                           "Year Built"]).copy()
lab["premium"] = lab["Neighborhood"].isin(BARRIOS_PREMIUM).astype(int)
lab["Area_x_premium"] = lab["Gr Liv Area"] * lab["premium"]
print("Casas en barrios premium:", int(lab["premium"].sum()), "de", len(lab))

**❓ Qué se quiere averiguar.** ¿Vale lo mismo un pie² en un barrio premium (`NridgHt`, `NoRidge`, `StoneBr`) que en el resto de Ames? Dicho en términos de la tasadora: ¿basta **una** tarifa por superficie para toda la ciudad?

- **Qué decide:** si se publica una sola tabla de precios por pie² o una por zona, y cuánto se pierde al ampliar una vivienda premium con la tarifa general. Eso es una interacción traducida a negocio: el efecto del tamaño **depende** de la zona.
- **Antes de mirar el resultado:** si `β₃ ≈ 0` con `p ≥ 0,05`, las dos rectas serían paralelas, una sola tarifa bastaría y el término sobraría del modelo. Si `β₃ > 0` y significativo, cada pie² rinde más en zona premium y la tarifa única **infravalora** esas casas. Como el modelo en nivel rechaza la homocedasticidad, la significancia se juzga con los SE robustos HC3, no con los clásicos.

**🔎 Qué hace este código.** Ajusta el modelo con interacción `SalePrice ~ Gr Liv Area + premium + (Gr Liv Area × premium)` (principio de jerarquía) y extrae la pendiente base, el coeficiente de la interacción con su p-valor, y la pendiente resultante en barrio premium.

In [ ]:
# Modelo con interacción zona × tamaño (principio de jerarquía: X1, dummy y su producto)
Xint = sm.add_constant(lab[["Gr Liv Area", "premium", "Area_x_premium"]])
modelo_interaccion = sm.OLS(lab["SalePrice"], Xint).fit()

beta_base = modelo_interaccion.params["Gr Liv Area"]
beta_inter = modelo_interaccion.params["Area_x_premium"]
p_inter = modelo_interaccion.pvalues["Area_x_premium"]
print("Pendiente base (barrio NO premium): %.1f USD/pie2" % beta_base)
print("Coeficiente de la interacción     : %+.1f USD/pie2  (p = %.2e)" % (beta_inter, p_inter))
print("Pendiente en barrio premium       : %.1f USD/pie2" % (beta_base + beta_inter))
print("R² del modelo con interacción     : %.4f" % modelo_interaccion.rsquared)

# Inferencia robusta HC3 del modelo con interacción (en nivel: BP rechaza homocedasticidad)
bp_inter = het_breuschpagan(modelo_interaccion.resid, modelo_interaccion.model.exog)
modelo_interaccion_hc3 = sm.OLS(lab["SalePrice"], Xint).fit(cov_type="HC3")
ic_int_hc3 = modelo_interaccion_hc3.conf_int()
robusta_int = pd.DataFrame({
    "coef": modelo_interaccion_hc3.params,
    "SE_HC3": modelo_interaccion_hc3.bse,
    "t_HC3": modelo_interaccion_hc3.tvalues,
    "p_HC3": modelo_interaccion_hc3.pvalues,
    "IC95_bajo": ic_int_hc3[0],
    "IC95_alto": ic_int_hc3[1],
})
print("\nBreusch-Pagan del modelo con interacción: LM = %.1f, p = %.2e -> %s"
      % (bp_inter[0], bp_inter[1], "usar HC3" if bp_inter[1] < 0.05 else "homocedástico"))
print("Inferencia robusta HC3 (interacción) — el coef no cambia, solo su SE/t/p/IC:")
print(robusta_int.round(4).to_string())

**📖 Lectura de la interacción.** El coeficiente de la interacción (~**+42,7 USD/pie²**) es el **extra de valor por pie²** en un barrio premium frente al resto, y su p-valor ≪ 0,001 confirma que es real: *el efecto del tamaño sobre el precio depende de la zona*. La pendiente pasa de ~88 a ~131 USD/pie² (casi 50 % más). No se interpreta la pendiente base aislada como «el efecto del tamaño» — con interacción la pendiente es `β₁ + β₃·premium`. Como el modelo de interacción en nivel también **rechaza la homocedasticidad** (Breusch-Pagan LM≈318, p≈0), la significancia se juzga con **SE robustos HC3**: el término de interacción sigue siendo significativo (`p_HC3 ≈ 1,2·10⁻⁷`) aunque su SE crece de ~5,9 a ~8,1 USD/pie²; la conclusión no cambia.

**❓ Qué se quiere averiguar.** Antes de entregar el modelo, ¿se pueden citar sus cinco coeficientes uno a uno en el informe, o solo sirve para predecir en bloque?

- **Qué decide:** qué frases del informe se sostienen. «Cada plaza de garaje añade X USD» solo es defendible si ese coeficiente no está inflado por colinealidad, y el ancho de los intervalos depende de si el error es constante.
- **Antes de mirar el resultado:** si algún `VIF` pasara de 5, ese coeficiente se retira del texto **aunque el modelo conserve su precisión predictiva**, porque atribuiría a una variable lo que comparte con otra. Si los cinco quedan por debajo de 5, los efectos son interpretables por separado. Y si Breusch-Pagan arroja `p < 0,05`, la incertidumbre se reporta con HC3.

**🔎 Qué hace este código.** Diagnostica el modelo múltiple de negocio: calcula el **VIF** de los 5 predictores, la prueba de **Breusch-Pagan** y el **Durbin-Watson**. Es el diagnóstico que decide si los coeficientes son interpretables y si hay que reportar errores robustos.

In [ ]:
# Diagnóstico del laboratorio: VIF del modelo múltiple de negocio + Breusch-Pagan + robustos
PREDS_LAB = ["Overall Qual", "Gr Liv Area", "Garage Cars", "Total Bsmt SF", "Year Built"]
Xlab = sm.add_constant(lab[PREDS_LAB])
modelo_lab = sm.OLS(lab["SalePrice"], Xlab).fit()

vif_lab = pd.DataFrame({
    "predictor": PREDS_LAB,
    "VIF": [variance_inflation_factor(Xlab.values, i + 1) for i in range(len(PREDS_LAB))],
})
print(vif_lab.round(3).to_string(index=False))

bp_lab = het_breuschpagan(modelo_lab.resid, modelo_lab.model.exog)
print("\nBreusch-Pagan del modelo de negocio: LM = %.1f, p = %.2e -> %s"
      % (bp_lab[0], bp_lab[1], "usar errores robustos (HC3)" if bp_lab[1] < 0.05 else "homocedástico"))
print("Durbin-Watson: %.3f" % durbin_watson(modelo_lab.resid))

**📖 Diagnóstico.** Todos los VIF < 5 (máx `Overall Qual ≈ 2,47`) ⇒ **sin multicolinealidad preocupante**; los coeficientes son interpretables. Breusch-Pagan rechaza homocedasticidad en el modelo en nivel ⇒ se reportarían los **errores robustos HC3**. El Durbin-Watson (≈ 1,68) aquí **sí es interpretable**: el CSV de De Cock viene **agrupado geográficamente** (orden por PID; ~86 % de filas contiguas comparten `Neighborhood`), de modo que el desvío respecto de 2 es **señal de dependencia espacial residual** (ρ̂ ≈ 0,16 intra-barrio), no ruido de archivo. Consecuencia: los SE clásicos **y** los HC3 son optimistas frente a esa dependencia; la inferencia honesta del modelo desplegado agrupa por barrio (hoja `se_cluster_barrio`; «Errores estándar robustos», capítulo 4.6).

**❓ Qué se quiere averiguar.** ¿Cuántos dólares se equivoca la tasación en una casa que el modelo **no** ha visto, y compensa recolectar cinco variables en vez de una?

- **Qué decide:** qué modelo se despliega. El RMSE está en USD, es el error medio por tasación, la única cifra de esta celda que un gerente puede comparar contra el margen de la operación.
- **Antes de mirar el resultado:** el modelo de solo superficie fija la referencia. Si el múltiple no bajara de ese RMSE, las cuatro variables extra costarían recolección sin ganancia. Si baja de forma apreciable **y** además se reduce la desviación entre pliegues, el modelo con más predictores supera al simple en dos frentes: presenta menor error y es más estable. Regla de la sesión: cuando el `R²` ajustado dentro de muestra y la CV discrepen, **prevalece la CV**.

**🔎 Qué hace este código.** Compara dos modelos por **CV 5-fold** en la misma escala (nivel USD, comparable por RMSE): Modelo A (solo superficie) vs. Modelo B (5 predictores). Reporta `R²` medio ± sd, `RMSE` medio ± sd y el `R²` ajustado in-sample de cada uno. Se usa `shuffle=True` porque la base de Ames viene **ordenada** (por `Order`/`PID`, correlacionado con el barrio): sin barajar, cada fold quedaría dominado por unas pocas zonas y el error entre folds saldría sesgado; `random_state=42` fija el barajado para que sea reproducible. **Nota de escala:** la elección de modelo se realiza en **nivel USD** para que el RMSE del simple y del múltiple sean comparables; el modelo que se despliega es el **log** (Sección 4), cuya generalización se confirma con una CV en escala log más abajo en esta misma celda.

**Capa docente — `n_jobs=1`.** `cross_val_score` puede repartir los folds entre varios procesos (`n_jobs=-1`). Al hacerlo, cada proceso hereda su propia configuración de hilos BLAS y los resultados **se ensamblan en el orden en que van terminando**, de modo que la media entre folds puede cambiar en los últimos decimales de una ejecución a otra y de un equipo a otro. Con `n_jobs=1` los cinco folds se ajustan **en serie, siempre en el mismo orden**, y la CV se vuelve reproducible bit a bit —requisito del bloque «Determinismo» del validador, que ejecuta la CV dos veces y exige resultado **idéntico**—. El costo es despreciable (cinco regresiones lineales); la ganancia es que el 0,8194 del contrato es siempre el mismo número.

In [ ]:
# Elección de modelo por CV 5-fold (misma escala: nivel USD, comparable por RMSE)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
y_lab = lab["SalePrice"].values

XA = lab[["Gr Liv Area"]].values                 # Modelo A: simple (solo superficie)
XB = lab[PREDS_LAB].values                        # Modelo B: múltiple (5 predictores)

def cv_resumen(X, y):
    # n_jobs=1: los 5 folds se ajustan EN SERIE y siempre en el mismo orden -> la media
    # entre folds es reproducible bit a bit (con n_jobs=-1 el ensamblado depende de que
    # proceso termine antes y la cifra baila en los ultimos decimales).
    r2 = cross_val_score(LinearRegression(), X, y, cv=kf, scoring="r2", n_jobs=1)
    rmse = -cross_val_score(LinearRegression(), X, y, cv=kf,
                            scoring="neg_root_mean_squared_error", n_jobs=1)
    return r2.mean(), r2.std(), rmse.mean(), rmse.std()

r2A, sdA, rmseA, rmsesdA = cv_resumen(XA, y_lab)
r2B, sdB, rmseB, rmsesdB = cv_resumen(XB, y_lab)

adjA = sm.OLS(y_lab, sm.add_constant(XA)).fit().rsquared_adj
adjB = sm.OLS(y_lab, sm.add_constant(XB)).fit().rsquared_adj

print("Modelo A (simple)   : CV R² = %.4f ± %.4f | RMSE = %.0f ± %.0f | R²adj = %.4f"
      % (r2A, sdA, rmseA, rmsesdA, adjA))
print("Modelo B (múltiple) : CV R² = %.4f ± %.4f | RMSE = %.0f ± %.0f | R²adj = %.4f"
      % (r2B, sdB, rmseB, rmsesdB, adjB))

# CV del modelo DESPLEGADO (log): confirma que R²≈0,847 no es sobreajuste (escala log, no comparable por RMSE)
Xlog = lab[PREDS_LAB].values
y_log_lab = np.log(lab["SalePrice"].values)
cv_r2_log = cross_val_score(LinearRegression(), Xlog, y_log_lab, cv=kf, scoring="r2", n_jobs=1)
cv_r2_log_mean, cv_r2_log_sd = cv_r2_log.mean(), cv_r2_log.std()
print("Modelo log (desplegado): CV R² (escala log) = %.4f ± %.4f  (in-sample 0,847)"
      % (cv_r2_log_mean, cv_r2_log_sd))

# --- Misma escala: el RMSE del modelo LOG llevado a USD con el SMEARING DE DUAN (1983) ---
# Un R2 en log (0,847) y uno en nivel (0,820) NO son comparables. La salida correcta es traer el
# modelo log a USD: E[y|X] = exp(Xb) x S, con S = media(exp(residuos)). Retransformar con exp() a
# secas devuelve la MEDIANA condicional, no la media: subestima el precio esperado (Jensen).
# El factor S se estima SOLO con el fold de entrenamiento (nunca con el de prueba).
_rmse_duan, _rmse_naive = [], []
for _tr, _te in kf.split(XB):
    _m = LinearRegression().fit(XB[_tr], np.log(y_lab[_tr]))
    _s_tr = float(np.mean(np.exp(np.log(y_lab[_tr]) - _m.predict(XB[_tr]))))   # smearing del fold
    _p_usd = np.exp(_m.predict(XB[_te]))
    _rmse_duan.append(np.sqrt(np.mean((y_lab[_te] - _p_usd * _s_tr) ** 2)))
    _rmse_naive.append(np.sqrt(np.mean((y_lab[_te] - _p_usd) ** 2)))
rmse_usd_log_cv = float(np.mean(_rmse_duan))
rmse_usd_log_cv_sd = float(np.std(_rmse_duan))
rmse_usd_log_cv_naive = float(np.mean(_rmse_naive))

_m_full = LinearRegression().fit(XB, np.log(y_lab))
factor_smearing = float(np.mean(np.exp(np.log(y_lab) - _m_full.predict(XB))))
rmse_usd_log_insample = float(np.sqrt(np.mean((y_lab - np.exp(_m_full.predict(XB)) * factor_smearing) ** 2)))

print("\nMISMA ESCALA (USD) — modelo log retransformado con el smearing de Duan:")
print("  factor de smearing S       = %.6f  (+%.2f %% sobre exp(pred))"
      % (factor_smearing, (factor_smearing - 1) * 100))
print("  RMSE CV (USD, con Duan)    = %.0f ± %.0f" % (rmse_usd_log_cv, rmse_usd_log_cv_sd))
print("  RMSE CV (USD, sin corregir)= %.0f   <- exp() a secas subestima el precio esperado"
      % rmse_usd_log_cv_naive)
print("  RMSE CV multiple en NIVEL  = %.0f   -> el log en USD mejora en %.0f USD (%.1f %%)"
      % (rmseB, rmseB - rmse_usd_log_cv, (rmseB - rmse_usd_log_cv) / rmseB * 100))

**📖 Decisión por CV.** El modelo múltiple reduce el RMSE fuera de muestra de ~54 500 a ~33 300 USD (−39 %) **y** es más estable (menor sd entre folds). R² ajustado in-sample y CV coinciden aquí en preferir el modelo grande; cuando difieran, **prevalece la CV** porque mide generalización.

**Comparar el log con el nivel: el smearing de Duan.** El `R²` del modelo en **nivel** (0,820) y el del modelo en **log** (0,847) **no se pueden comparar**: miden la varianza explicada de variables distintas (dólares y logaritmos de dólares). La comparación legítima exige traer ambos a la **misma escala**, y eso significa llevar el modelo log a **USD**. La retransformación ingenua `exp(X β̂)` **no** devuelve el precio *esperado* sino la **mediana** condicional: por la desigualdad de Jensen, `E[y] = E[exp(log y)] ≥ exp(E[log y])`, así que sistemáticamente **subestima**. La corrección estándar es el **factor de smearing de Duan (1983)**: `S = (1/n)·Σ exp(eᵢ)` sobre los residuales del modelo en log, y la predicción en dólares pasa a ser `ŷ = exp(X β̂)·S`. En Ames `S ≈ 1,0119` (+1,2 %), y el factor se estima **en cada fold de entrenamiento**, nunca con el de prueba. El resultado, ya comparable: el modelo **log retransformado** rinde **RMSE ≈ 27 282 USD** en CV 5-fold, frente a **33 308 USD** del múltiple en nivel — **≈ 6 027 USD menos de error por tasación (−18,1 %)**. Es decir: el modelo en log no solo se interpreta mejor (en porcentajes), sino que **predice mejor en dólares**; lo que faltaba era ponerlos en la misma unidad. Todo se registra en la hoja `retransformacion_duan` del Excel.

**Recomendación de negocio.** Para valuar una vivienda en Ames se recomienda el modelo múltiple (calidad + superficie + garaje + sótano + año), reportando la incertidumbre con errores robustos y reconociendo que el **valor del pie² es mayor en barrios premium** (interacción). Un β significativo mide **asociación condicionada**, no causa (la guía de supuestos de la sesión, Parte 1.2).

**Reconciliación de los R² del múltiple.** Conviven tres cifras que miden magnitudes distintas: (i) **R² in-sample = 0,847** del modelo **log** desplegado (Sección 4); (ii) **CV R² ≈ 0,846** de ese mismo modelo **log** (recién calculada), casi igual al in-sample, así que 0,847 **no es sobreajuste**; y (iii) **CV R² ≈ 0,819 (≈0,82)** del múltiple **en nivel USD**, la escala en la que se comparan los RMSE para **elegir** modelo. El 0,82 es, por tanto, un R² de **validación cruzada en nivel**, no un R² in-sample, y la CV del modelo efectivamente desplegado (log) también lo respalda.

## Transversal — Exportación a Excel y figuras (Sección 6 del cuaderno)

**Convención del curso:** los resultados y pruebas del modelo se vuelcan a `resultados/S04_resultados.xlsx` (openpyxl) y las **figuras de resultados se generan LEYENDO ese Excel** (nunca desde objetos en memoria). Las figuras de **EDA/diagnóstico de datos crudos** se trazan directamente de los datos (Sección 8).

**🔎 Qué hace este código.** Vuelca al Excel de contrato las **6 hojas** de la sesión —`regresion_ames`, `diagnostico_vif`, `supuestos`, `cv_modelos`, `interaccion`, `coeficientes_multiple`— más **11 hojas auxiliares declaradas** —`inferencia_robusta`, `cv_reconciliacion`, `bootstrap_ic`, `supuestos_panel`, `conversion_monetaria`, `cifras_titulares`, `retransformacion_duan`, `se_cluster_barrio` (SE agrupados por barrio + experimento del orden del DW), `interaccion_mediana` (el efecto premium evaluado en la superficie mediana real, 1 442 pie²) y `modelos_drills` (los modelos paralelos que usan los drills 1–3) y `dummy_log` (la demo ejecutable de la P7 del control corto: `log(SalePrice) ~ Gr Liv Area + Garage Cars + Total Bsmt SF + premium`, β_premium ≈ 0,1535 ⇒ e^β − 1 ≈ +16,6 %)— con los resultados ya calculados. `cifras_titulares` reúne las cifras que la guía cita como titulares y que hasta ahora no tenían celda propia (n de cada modelo, correlación tras la limpieza, pendiente simple, F global, R² ajustado, R² de la interacción, distancia de Cook, asimetría y λ de Box-Cox, VIF de la demo de garaje, y las conversiones del cierre), para que **toda** cifra publicada tenga trazabilidad en el contrato. **Es la única celda que escribe el Excel;** las figuras de resultados y las verificaciones se generan leyéndolo.

**Capa docente — `rcond=None` en `np.linalg.lstsq`.** Cada réplica del bootstrap resuelve un sistema de mínimos cuadrados por SVD. `rcond` es el **umbral relativo** por debajo del cual un valor singular se considera cero (columna redundante) y se descarta. `numpy` cambió su valor por defecto: el antiguo dependía del tamaño de la matriz (`max(M,N)·eps`) y el nuevo, más conservador, es `eps` de la máquina; llamar sin especificarlo emite un `FutureWarning` y —lo importante— **el corte podría diferir entre versiones de numpy**, cambiando qué se trata como colinealidad numérica. Fijar `rcond=None` **declara el comportamiento nuevo de forma explícita**, con lo que las 2000 réplicas resuelven todas con el mismo criterio y el IC bootstrap es idéntico entre máquinas. **Si no estuviera:** el resultado sería el mismo hoy y podría no serlo tras una actualización de `numpy`, justo la deriva silenciosa que el pineado de versiones intenta evitar.

In [ ]:
# --- Construir el libro Excel con el contrato exacto de hojas ---
from openpyxl import Workbook
from statsmodels.stats.stattools import jarque_bera
wb = Workbook()

# Hoja 1: regresion_ames (targets #1-#4)
ws = wb.active
ws.title = "regresion_ames"
ws["A1"] = "metrica"; ws["B1"] = "valor"
ws["A2"] = "corr_saleprice_grlivarea"; ws["B2"] = float(corr_saleprice_grlivarea)
ws["A3"] = "n_atipicas_gt4000";        ws["B3"] = int(n_atipicas_gt4000)
ws["A4"] = "r2_simple_limpio";         ws["B4"] = float(r2_simple_limpio)
ws["A5"] = "r2_multiple_log";          ws["B5"] = float(r2_multiple_log)

# Hoja 2: diagnostico_vif (VIF por predictor del modelo múltiple)
ws2 = wb.create_sheet("diagnostico_vif")
ws2.append(["predictor", "VIF"])
Xvif = sm.add_constant(mult[PREDS])
for i, c in enumerate(PREDS):
    ws2.append([c, float(variance_inflation_factor(Xvif.values, i + 1))])

# Hoja 3: supuestos (Breusch-Pagan, Durbin-Watson, SE clásico vs HC3)
ws3 = wb.create_sheet("supuestos")
ws3.append(["prueba", "estadistico", "valor"])
bp_simple = het_breuschpagan(modelo_simple.resid, modelo_simple.model.exog)
ols_lvl = sm.OLS(clean["SalePrice"], sm.add_constant(clean[["Gr Liv Area"]])).fit()
hc3_lvl = sm.OLS(clean["SalePrice"], sm.add_constant(clean[["Gr Liv Area"]])).fit(cov_type="HC3")
ws3.append(["breusch_pagan_nivel", "LM", float(bp_simple[0])])
ws3.append(["breusch_pagan_nivel", "p_valor", float(bp_simple[1])])
ws3.append(["durbin_watson_multiple", "DW", float(durbin_watson(modelo_multiple.resid))])
ws3.append(["SE_grlivarea", "clasico", float(ols_lvl.bse["Gr Liv Area"])])
ws3.append(["SE_grlivarea", "robusto_HC3", float(hc3_lvl.bse["Gr Liv Area"])])

# Hoja 4: cv_modelos (2 modelos por CV 5-fold + R² ajustado in-sample)
ws4 = wb.create_sheet("cv_modelos")
ws4.append(["modelo", "cv_r2_medio", "cv_r2_sd", "cv_rmse_medio", "cv_rmse_sd", "r2_ajustado_insample"])
ws4.append(["simple_grlivarea", float(r2A), float(sdA), float(rmseA), float(rmsesdA), float(adjA)])
ws4.append(["multiple_5preds", float(r2B), float(sdB), float(rmseB), float(rmsesdB), float(adjB)])

# Hoja 5: interaccion (coef de la interacción zona × tamaño + lectura)
ws5 = wb.create_sheet("interaccion")
ws5.append(["concepto", "valor"])
ws5.append(["pendiente_base_no_premium", float(beta_base)])
ws5.append(["coef_interaccion_area_x_premium", float(beta_inter)])
ws5.append(["pendiente_premium", float(beta_base + beta_inter)])
ws5.append(["p_valor_interaccion", float(p_inter)])
ws5.append(["lectura", "El valor del pie2 es mayor en barrios premium: el efecto del tamano depende de la zona"])

# Hoja 6: coeficientes_multiple (coef, efecto %, SE y p-valor del modelo múltiple en log)
ws6 = wb.create_sheet("coeficientes_multiple")
ws6.append(["termino", "coef", "efecto_pct", "SE", "p_valor"])
for _t in modelo_multiple.params.index:
    _b = float(modelo_multiple.params[_t])
    _efecto = "" if _t == "const" else float((np.exp(_b) - 1) * 100)
    ws6.append([_t, _b, _efecto, float(modelo_multiple.bse[_t]), float(modelo_multiple.pvalues[_t])])

# Hoja 7: inferencia_robusta (SE/t/p/IC 95% HC3 del modelo log y de la interacción; NO altera las 6 hojas de contrato)
ws7 = wb.create_sheet("inferencia_robusta")
ws7.append(["modelo", "termino", "coef", "se_hc3", "t_hc3", "p_hc3", "ci95_bajo_hc3", "ci95_alto_hc3"])
_ic_log = modelo_multiple_hc3.conf_int()
for _t in modelo_multiple_hc3.params.index:
    ws7.append(["log_multiple", _t, float(modelo_multiple_hc3.params[_t]),
                float(modelo_multiple_hc3.bse[_t]), float(modelo_multiple_hc3.tvalues[_t]),
                float(modelo_multiple_hc3.pvalues[_t]), float(_ic_log.loc[_t, 0]), float(_ic_log.loc[_t, 1])])
_ic_int = modelo_interaccion_hc3.conf_int()
for _t in modelo_interaccion_hc3.params.index:
    ws7.append(["interaccion", _t, float(modelo_interaccion_hc3.params[_t]),
                float(modelo_interaccion_hc3.bse[_t]), float(modelo_interaccion_hc3.tvalues[_t]),
                float(modelo_interaccion_hc3.pvalues[_t]), float(_ic_int.loc[_t, 0]), float(_ic_int.loc[_t, 1])])

# Hoja 8: cv_reconciliacion (los tres R2 del múltiple: log in-sample, log CV, nivel CV; NO altera el contrato)
ws8 = wb.create_sheet("cv_reconciliacion")
ws8.append(["concepto", "escala", "valor", "nota"])
ws8.append(["r2_insample", "log", float(r2_multiple_log), "R2 in-sample del modelo desplegado (log)"])
ws8.append(["cv_r2_medio", "log", float(cv_r2_log_mean), "CV 5-fold del modelo desplegado (log): 0,847 no es sobreajuste"])
ws8.append(["cv_r2_sd", "log", float(cv_r2_log_sd), "sd entre folds de la CV log"])
ws8.append(["cv_r2_medio", "nivel", float(r2B), "CV 5-fold en nivel USD: escala comparable por RMSE para elegir modelo"])
ws8.append(["r2adj_insample", "nivel", float(adjB), "R2 ajustado in-sample del multiple en nivel"])

# Hoja 9: bootstrap_ic — IC 95% no parametrico (remuestreo de casos) de la cifra insignia.
# Ruta INDEPENDIENTE del HC3: no supone normalidad. NO altera las 8 hojas previas.
_Xboot = sm.add_constant(mult[PREDS]).to_numpy(float)      # [const, Overall Qual, Gr Liv Area, ...]
_yboot = np.log(mult["SalePrice"].to_numpy(float))
_nboot = _Xboot.shape[0]
_col_oq = 1                                                # Overall Qual = columna 1 tras add_constant
_B_BOOT = 2000
_rng_boot = np.random.default_rng(42)                      # PCG64 reproducible, independiente del estado global
_r2_boot = np.empty(_B_BOOT); _coef_oq_boot = np.empty(_B_BOOT)
for _b in range(_B_BOOT):
    _idx = _rng_boot.integers(0, _nboot, _nboot)           # remuestreo de casos con reemplazo
    _Xb = _Xboot[_idx]; _yb = _yboot[_idx]
    _beta, *_rest = np.linalg.lstsq(_Xb, _yb, rcond=None)
    _res = _yb - _Xb @ _beta
    _r2_boot[_b] = 1 - (_res @ _res) / (((_yb - _yb.mean()) ** 2).sum())
    _coef_oq_boot[_b] = _beta[_col_oq]
_r2_lo, _r2_hi = np.percentile(_r2_boot, [2.5, 97.5])
_coq_lo, _coq_hi = np.percentile(_coef_oq_boot, [2.5, 97.5])
_eff_boot = (np.exp(_coef_oq_boot) - 1) * 100
_eff_lo, _eff_hi = np.percentile(_eff_boot, [2.5, 97.5])
_r2_point = float(modelo_multiple.rsquared)
_coq_point = float(modelo_multiple.params["Overall Qual"])
_eff_point = float((np.exp(_coq_point) - 1) * 100)

ws9 = wb.create_sheet("bootstrap_ic")
ws9.append(["cifra", "escala", "punto", "ic95_bajo", "ic95_alto", "metodo", "B", "semilla"])
ws9.append(["r2_multiple_log", "R2", _r2_point, float(_r2_lo), float(_r2_hi), "bootstrap_casos_percentil", _B_BOOT, 42])
ws9.append(["coef_overall_qual", "log", _coq_point, float(_coq_lo), float(_coq_hi), "bootstrap_casos_percentil", _B_BOOT, 42])
ws9.append(["efecto_overall_qual_pct", "pct", _eff_point, float(_eff_lo), float(_eff_hi), "bootstrap_casos_percentil", _B_BOOT, 42])

# Hoja 10: supuestos_panel — tabla de supuestos VERIFICADA (prueba, umbral, veredicto,
# consecuencia). Deriva de la guía de supuestos de la sesión. NO altera las hojas previas.
_bp_log = het_breuschpagan(modelo_multiple.resid, modelo_multiple.model.exog)
_dw_log = float(durbin_watson(modelo_multiple.resid))
_jb_log = jarque_bera(modelo_multiple.resid)               # (JB, p, skew, kurtosis)
_vif_vals = [float(variance_inflation_factor(Xvif.values, i + 1)) for i in range(len(PREDS))]
_vif_max = max(_vif_vals); _vif_arg = PREDS[int(np.argmax(_vif_vals))]

ws10 = wb.create_sheet("supuestos_panel")
ws10.append(["supuesto", "prueba", "estadistico", "valor", "umbral", "veredicto", "consecuencia"])
ws10.append(["1.3 Homocedasticidad (log)", "Breusch-Pagan", "LM", float(_bp_log[0]), "p<0,05 rechaza",
             "rechaza (p=%.1e)" % _bp_log[1], "usar SE robustos HC3 e IC bootstrap; no cambia el coef"])
# Durbin-Watson: el CSV de De Cock viene AGRUPADO GEOGRAFICAMENTE (orden por PID; ~86 % de
# filas contiguas comparten barrio), asi que el orden ES informativo y el DW = 1,68 SI se
# interpreta: senal de dependencia espacial residual (rho ~ 0,16 intra-barrio). Verificado:
# barajado (semilla 42) el DW vuelve a ~1,97; ordenado por barrio reproduce ~1,70 (ver hoja
# se_cluster_barrio). Consecuencia: SE clasicos y HC3 optimistas -> SE agrupados por barrio.
ws10.append(["1.4 No autocorrelacion (log)", "Durbin-Watson", "DW", _dw_log,
             "~2 sano solo si el orden es informativo; aqui lo es (agrupado por barrio)",
             "senal de dependencia espacial (orden geografico por PID); SE por barrio en hoja se_cluster_barrio",
             "SE clasicos y HC3 optimistas ante dependencia intra-barrio; remedio S04: SE cluster por barrio; tratamiento espacial completo -> S11+"])
ws10.append(["1.5 Multicolinealidad (estrella)", "VIF maximo (%s)" % _vif_arg, "VIF", _vif_max, ">5 preocupa; >10 severa",
             "sin problema (<5)" if _vif_max < 5 else "actuar", "coeficientes interpretables con confianza"])
ws10.append(["2.1 Normalidad de residuos (log)", "Jarque-Bera", "JB", float(_jb_log[0]), "p<0,05 rechaza",
             "rechaza (p=%.1e)" % _jb_log[1], "n~2923: TCL protege; los IC HC3/bootstrap no exigen normalidad"])

print("\nIC bootstrap 95%% (B=%d, semilla 42) — cifra insignia:" % _B_BOOT)
print("  R2 multiple (log)      : %.4f  IC [%.4f, %.4f]" % (_r2_point, _r2_lo, _r2_hi))
print("  efecto Overall Qual (%%): %.2f  IC [%.2f, %.2f]" % (_eff_point, _eff_lo, _eff_hi))

# Hoja 11: conversion_monetaria - la mediana de SalePrice (casa tipica: base de todas las
# conversiones %->USD del cierre economico) y la traduccion de los efectos % a USD. Deriva de
# la base y de las hojas previas; NO altera las 10 hojas de contrato.
_mediana_usd = float(ames["SalePrice"].median())
_ef_oq_pct = float((np.exp(modelo_multiple.params["Overall Qual"]) - 1) * 100)
_ef_gc_pct = float((np.exp(modelo_multiple.params["Garage Cars"]) - 1) * 100)
_ic_oq = modelo_multiple_hc3.conf_int().loc["Overall Qual"]
_ef_oq_lo_pct = float((np.exp(_ic_oq[0]) - 1) * 100)
_ef_oq_hi_pct = float((np.exp(_ic_oq[1]) - 1) * 100)
ws11 = wb.create_sheet("conversion_monetaria")
ws11.append(["concepto", "valor_usd", "base", "nota"])
ws11.append(["mediana_saleprice", _mediana_usd, "SalePrice.median() (n=%d)" % len(ames),
             "casa tipica: base de las conversiones porcentaje->USD del cierre economico"])
ws11.append(["overall_qual_usd_por_punto", _mediana_usd * _ef_oq_pct / 100,
             "mediana x efecto porcentual de Overall Qual",
             "+%.1f%% por punto de calidad sobre la casa mediana" % _ef_oq_pct])
ws11.append(["overall_qual_usd_ic_bajo", _mediana_usd * _ef_oq_lo_pct / 100,
             "mediana x IC95 HC3 bajo", "extremo inferior del IC robusto por punto de calidad"])
ws11.append(["overall_qual_usd_ic_alto", _mediana_usd * _ef_oq_hi_pct / 100,
             "mediana x IC95 HC3 alto", "extremo superior del IC robusto por punto de calidad"])
ws11.append(["garage_cars_usd_por_plaza", _mediana_usd * _ef_gc_pct / 100,
             "mediana x efecto porcentual de Garage Cars",
             "+%.1f%% por plaza de garaje sobre la casa mediana" % _ef_gc_pct])

# Hoja 12: cifras_titulares - cifras que la guia/deck citan como titulares y que no tenian celda
# propia en el contrato (n de cada modelo, correlacion limpia, pendiente simple, F global, R2
# ajustado, R2 de la interaccion, Cook, asimetria/Box-Cox, VIF de la demo de garaje y las
# conversiones del cierre). Hoja NUEVA y declarada: no altera ninguna hoja ni columna previa.
# Cada fila se re-deriva por una via INDEPENDIENTE en el material de referencia de la sesión.
_infl_log = modelo_multiple.get_influence()
_cook_log = _infl_log.cooks_distance[0]
_lev_log = _infl_log.hat_matrix_diag
_stud_log = _infl_log.resid_studentized_internal
_umb_log = 4 / len(mult)
_p_log = len(PREDS) + 1
_cook_did = mc.get_influence().cooks_distance[0]      # modelo didactico de 2 predictores (celda 31)
_umb_did = 4 / len(dem)

_precio_dem = dem["SalePrice"].to_numpy(float)
_g_demo = ames.dropna(subset=["Garage Cars", "Garage Area", "Gr Liv Area", "Overall Qual"])
_Xg_demo = sm.add_constant(_g_demo[["Garage Cars", "Garage Area", "Gr Liv Area", "Overall Qual"]])
_vif_demo = {c: float(variance_inflation_factor(_Xg_demo.values, i))
             for i, c in enumerate(_Xg_demo.columns)}
_otros_oq = [p for p in PREDS if p != "Overall Qual"]
_r2j_oq = float(sm.OLS(mult["Overall Qual"], sm.add_constant(mult[_otros_oq])).fit().rsquared)
_b_gla = float(modelo_multiple.params["Gr Liv Area"])
_ef_oq = float((np.exp(modelo_multiple.params["Overall Qual"]) - 1) * 100)

ws12 = wb.create_sheet("cifras_titulares")
ws12.append(["concepto", "valor", "modelo_o_base", "nota"])
for _fila in [
    ("n_ames_total", len(ames), "AmesHousing.csv (De Cock)",
     "base original: 2930 ventas x 82 variables"),
    ("n_clean_sin_atipicas", len(clean), "Gr Liv Area <= 4000",
     "tras remover las 5 atipicas de De Cock"),
    ("n_multiple_log", len(mult), "log(SalePrice) ~ 5 predictores",
     "n del modelo desplegado (dropna de los 5 predictores)"),
    ("n_laboratorio_interaccion", len(lab), "SalePrice ~ Gr Liv Area * premium",
     "n del laboratorio de negocio"),
    ("n_casas_premium", int(lab["premium"].sum()), "Neighborhood in NridgHt/NoRidge/StoneBr",
     "casas en barrio premium dentro del laboratorio"),
    ("corr_limpia_post_remocion", float(clean["SalePrice"].corr(clean["Gr Liv Area"])),
     "clean", "la correlacion sube de 0,7068 a 0,7195 al remover las 5"),
    ("pendiente_simple_usd_pie2", float(modelo_simple.params["Gr Liv Area"]),
     "SalePrice ~ Gr Liv Area (clean)",
     "USD por pie2 de la recta simple; no es la base 88,05 del modelo con interaccion"),
    ("coef_grlivarea_solo", float(m_bruto.params["Gr Liv Area"]),
     "SalePrice ~ Gr Liv Area (dem)", "coeficiente bruto: arrastra el efecto de la calidad"),
    ("coef_grlivarea_con_calidad", float(m_parcial.params["Gr Liv Area"]),
     "SalePrice ~ Gr Liv Area + Overall Qual",
     "coeficiente parcial: encoge al controlar por calidad"),
    ("f_global_multiple_log", float(modelo_multiple.fvalue), "modelo log",
     "significancia conjunta del modelo"),
    ("gl_denominador_f", int(modelo_multiple.df_resid), "n - k (k = 6 con const)",
     "grados de libertad del denominador de la F"),
    ("f_pvalor_multiple_log", float(modelo_multiple.f_pvalue), "modelo log",
     "p(F) ~ 0: el conjunto es masivamente significativo"),
    ("f_critica_5pct", float(stats.f.ppf(0.95, len(PREDS), modelo_multiple.df_resid)),
     "F(0,95; 5, gl)", "umbral critico al 5 %: la F observada esta muy por encima"),
    ("r2_ajustado_multiple_log", float(modelo_multiple.rsquared_adj), "modelo log",
     "casi identico al R2 crudo: con n grande el castigo es invisible"),
    ("r2_interaccion_nivel", float(modelo_interaccion.rsquared),
     "SalePrice ~ Gr Liv Area * premium", "R2 del modelo con interaccion"),
    ("se_clasico_interaccion", float(modelo_interaccion.bse["Area_x_premium"]),
     "coef Area_x_premium, SE clasico",
     "pasa a 8,0774 con HC3 (hoja inferencia_robusta): +37 % de incertidumbre declarada"),
    ("cook_umbral_4n_log", float(_umb_log), "modelo log (n=2923)", "umbral sensible 4/n"),
    ("cook_candidatos_log", int((_cook_log > _umb_log).sum()), "modelo log (n=2923)",
     "candidatos a inspeccion, no a borrado (~6-7 % de los datos)"),
    ("cook_maximo_log", float(_cook_log.max()), "modelo log",
     "muy por debajo de 1: ninguna observacion domina el ajuste"),
    ("leverage_altos_log", int((_lev_log > 2 * _p_log / len(mult)).sum()),
     "h_i > 2p/n (modelo log)", "puntos con palanca alta"),
    ("estudentizados_gt3_log", int((np.abs(_stud_log) > 3).sum()),
     "|residuo estudentizado| > 3 (modelo log)", "residuos grandes"),
    ("cook_umbral_4n_didactico", float(_umb_did),
     "SalePrice ~ Gr Liv Area + Overall Qual (n=2925)", "mismo umbral sobre el modelo didactico"),
    ("cook_candidatos_didactico", int((_cook_did > _umb_did).sum()),
     "modelo didactico (n=2925)", "el conteo cambia porque cambia el modelo, no el criterio"),
    ("cook_maximo_didactico", float(_cook_did.max()), "modelo didactico", "tambien muy por debajo de 1"),
    ("asimetria_saleprice", float(stats.skew(_precio_dem)), "dem",
     "cola derecha larga: asimetria ~ 1,6"),
    ("asimetria_log_saleprice", float(stats.skew(np.log(_precio_dem))), "dem",
     "el log la lleva a ~ 0"),
    ("boxcox_lambda", float(stats.boxcox(_precio_dem)[1]), "dem",
     "lambda ~ 0 confirma que el log es la transformacion adecuada"),
    ("vif_demo_garage_cars", _vif_demo["Garage Cars"], "demo: Garage Cars + Garage Area juntas",
     "colinealidad en vivo: el VIF sube a ~5"),
    ("vif_demo_garage_area", _vif_demo["Garage Area"], "demo: Garage Cars + Garage Area juntas",
     "las dos miden lo mismo: el modelo no puede separarlas"),
    ("r2_auxiliar_overall_qual", _r2j_oq, "Overall Qual ~ los otros 4 predictores",
     "VIF = 1/(1-0,5957) = 2,4735: el VIF no es un numero magico de la libreria"),
    ("efecto_grlivarea_pct_100pie2", float((np.exp(100 * _b_gla) - 1) * 100), "modelo log",
     "+0,028 % por pie2 equivale a ~ +2,8 % por cada 100 pie2"),
    ("ganancia_pendiente_premium_pct",
     float((beta_base + beta_inter) / beta_base - 1) * 100, "interaccion",
     "130,75 vs 88,05: ~ +49 % por pie2 en zona premium"),
    ("reduccion_rmse_cv_pct", float((rmseA - rmseB) / rmseA * 100), "CV nivel: simple vs multiple",
     "-39 % de error fuera de muestra"),
    ("ahorro_rmse_usd_por_tasacion", float(rmseA - rmseB), "CV nivel: 54 506 - 33 308",
     "USD de error evitado por tasacion"),
    ("overall_qual_usd_casa_400k", 400000.0 * _ef_oq / 100,
     "400 000 USD x efecto porcentual de Overall Qual",
     "el % escala con el precio de la casa sin reestimar el modelo"),
    ("inflacion_se_hc3_overall_qual_pct",
     float(modelo_multiple_hc3.bse["Overall Qual"] / modelo_multiple.bse["Overall Qual"] - 1) * 100,
     "SE HC3 / SE clasico - 1", "la heterocedasticidad inflaba la significancia"),
    ("inflacion_se_hc3_garage_cars_pct",
     float(modelo_multiple_hc3.bse["Garage Cars"] / modelo_multiple.bse["Garage Cars"] - 1) * 100,
     "SE HC3 / SE clasico - 1", "mismo coeficiente, incertidumbre honesta"),
]:
    ws12.append(list(_fila))

# Hoja 13: retransformacion_duan - el RMSE del modelo LOG llevado a USD con el factor de smearing
# de Duan (1983). Es la UNICA comparacion legitima contra el RMSE del modelo en nivel: un R2 en log
# (0,847) y uno en nivel (0,820) miden la varianza de variables distintas y NO son comparables.
# Hoja NUEVA y declarada: no altera ninguna hoja ni columna previa.
ws13 = wb.create_sheet("retransformacion_duan")
ws13.append(["concepto", "valor", "escala", "nota"])
for _fila in [
    ("factor_smearing_duan", factor_smearing, "factor",
     "S = media(exp(residuos del modelo log)); E[y|X] = exp(Xb) x S (Duan 1983)"),
    ("rmse_usd_log_insample", rmse_usd_log_insample, "USD",
     "RMSE in-sample del modelo desplegado (log) retransformado a USD"),
    ("rmse_usd_log_cv", rmse_usd_log_cv, "USD",
     "CIFRA TITULAR: RMSE de CV 5-fold del modelo log en USD, comparable con cv_modelos"),
    ("rmse_usd_log_cv_sd", rmse_usd_log_cv_sd, "USD", "desviacion entre los 5 folds"),
    ("rmse_usd_log_cv_sin_smearing", rmse_usd_log_cv_naive, "USD",
     "contrafactual: exp() a secas, sin corregir el sesgo de retransformacion"),
    ("sesgo_retransformacion_usd", rmse_usd_log_cv_naive - rmse_usd_log_cv, "USD",
     "lo que cuesta olvidar el smearing"),
    ("mejora_vs_nivel_usd", rmseB - rmse_usd_log_cv, "USD",
     "el log retransformado predice mejor que el multiple en nivel (33 308 USD)"),
    ("mejora_vs_nivel_pct", (rmseB - rmse_usd_log_cv) / rmseB * 100, "%",
     "reduccion relativa del error frente al modelo en nivel, ya en la misma escala"),
]:
    ws13.append(list(_fila))

print("\nRetransformacion a USD (smearing de Duan): S = %.6f | RMSE CV log en USD = %.0f "
      "vs %.0f del modelo en nivel (-%.1f %%)"
      % (factor_smearing, rmse_usd_log_cv, rmseB, (rmseB - rmse_usd_log_cv) / rmseB * 100))


# Hoja 14: se_cluster_barrio - SE agrupados por barrio (cluster-robust por Neighborhood) del
# modelo multiple log + experimento del orden del Durbin-Watson. El CSV de De Cock viene
# AGRUPADO GEOGRAFICAMENTE (orden por PID: ~86 % de filas contiguas comparten Neighborhood),
# asi que el DW = 1,68 SI es interpretable: dependencia espacial residual (rho ~ 0,16
# intra-barrio). HC3 corrige heterocedasticidad, NO dependencia: el SE honesto agrupa por
# barrio. Hoja NUEVA y declarada: no altera ninguna hoja previa.
modelo_multiple_cluster = sm.OLS(y_log, Xm).fit(
    cov_type="cluster", cov_kwds={"groups": mult["Neighborhood"]})
_ic_clu = modelo_multiple_cluster.conf_int()
ws14 = wb.create_sheet("se_cluster_barrio")
ws14.append(["termino", "coef", "se_clasico", "se_hc3", "se_cluster", "t_cluster",
             "p_cluster", "ic95_bajo_cluster", "ic95_alto_cluster"])
for _t in modelo_multiple.params.index:
    ws14.append([_t, float(modelo_multiple.params[_t]), float(modelo_multiple.bse[_t]),
                 float(modelo_multiple_hc3.bse[_t]), float(modelo_multiple_cluster.bse[_t]),
                 float(modelo_multiple_cluster.tvalues[_t]),
                 float(modelo_multiple_cluster.pvalues[_t]),
                 float(_ic_clu.loc[_t, 0]), float(_ic_clu.loc[_t, 1])])

# Experimento del orden (mismas filas, bloque concepto|valor|nota): que parte del DW es orden
# y que parte es estructura espacial. Determinista (semilla 42).
_res_dw = modelo_multiple.resid.to_numpy(float)
_barrio_dw = mult["Neighborhood"].to_numpy()
_dw_de = lambda r: float(np.sum(np.diff(r) ** 2) / np.sum(r ** 2))
_dw_orig = _dw_de(_res_dw)
_dw_baraja = _dw_de(_res_dw[np.random.default_rng(42).permutation(len(_res_dw))])
_dw_barrio = _dw_de(_res_dw[np.argsort(_barrio_dw, kind="quicksort")])
_contig_pct = float((ames["Neighborhood"].values[1:] == ames["Neighborhood"].values[:-1]).mean() * 100)
_ef_oq_clu_lo = float((np.exp(_ic_clu.loc["Overall Qual", 0]) - 1) * 100)
_ef_oq_clu_hi = float((np.exp(_ic_clu.loc["Overall Qual", 1]) - 1) * 100)
ws14.append(["concepto", "valor", "nota"])
for _fila in [
    ("n_barrios_cluster", float(mult["Neighborhood"].nunique()),
     "clusters (Neighborhood) del modelo log n=2923"),
    ("dw_original", _dw_orig, "orden del CSV (agrupado por PID/barrio)"),
    ("rho_orden_original", 1.0 - _dw_orig / 2.0,
     "rho = 1 - DW/2: dependencia espacial residual intra-barrio"),
    ("dw_barajado_semilla42", _dw_baraja,
     "residuos permutados con default_rng(42): sin orden informativo el DW vuelve a ~2"),
    ("dw_ordenado_por_barrio", _dw_barrio,
     "residuos ordenados por Neighborhood (argsort quicksort): reproduce el desvio -> la senal es espacial"),
    ("contiguidad_barrio_pct", _contig_pct,
     "% de filas contiguas del CSV (2930) que comparten barrio (~6,7 % si el orden fuera azar)"),
    ("efecto_oq_ic_bajo_cluster_pct", _ef_oq_clu_lo,
     "IC 95 % cluster del efecto de Overall Qual (%): mas ancho que el HC3 [9,88; 11,92]"),
    ("efecto_oq_ic_alto_cluster_pct", _ef_oq_clu_hi, "idem, extremo superior"),
    ("overall_qual_usd_ic_bajo_cluster", _mediana_usd * _ef_oq_clu_lo / 100,
     "IC en USD sobre la casa mediana (160 000): comparar con HC3 [15 804; 19 066]"),
    ("overall_qual_usd_ic_alto_cluster", _mediana_usd * _ef_oq_clu_hi / 100, "idem, extremo superior"),
]:
    ws14.append(list(_fila))

# Hoja 15: interaccion_mediana - el efecto premium del modelo con interaccion evaluado en la
# SUPERFICIE MEDIANA real del stock (Gr Liv Area = 1442 pie2), no solo en 2000 pie2 (~p80).
# El contrafactual completo del modelo a una superficie A es beta_premium + beta_interaccion*A;
# el +85 400 publicado es SOLO la via de la pendiente (42,70 x 2000). Hoja NUEVA declarada.
_b_prem = float(modelo_interaccion.params["premium"])
_med_gla = float(ames["Gr Liv Area"].median())
ws15 = wb.create_sheet("interaccion_mediana")
ws15.append(["concepto", "valor", "nota"])
for _fila in [
    ("mediana_grlivarea_base", _med_gla, "mediana de Gr Liv Area en la base completa (2930): 1442 pie2"),
    ("mediana_grlivarea_modelo", float(lab["Gr Liv Area"].median()),
     "mediana en el laboratorio n=2923 (1441): la casa tipica NO mide 2000 pie2 (2000 ~ p80)"),
    ("beta_premium_usd", _b_prem,
     "efecto de nivel de la dummy premium (hoja interaccion; no significativo por si solo)"),
    ("beta_interaccion_usd_pie2", float(beta_inter),
     "diferencia de pendiente premium vs estandar (hoja interaccion)"),
    ("efecto_premium_total_mediana_usd", _b_prem + float(beta_inter) * _med_gla,
     "beta_premium + beta_interaccion x 1442: diferencia total predicha entre zonas en la casa mediana"),
    ("efecto_premium_pendiente_mediana_usd", float(beta_inter) * _med_gla,
     "solo la via de la pendiente en la mediana (42,70 x 1442)"),
    ("efecto_premium_total_2000_usd", _b_prem + float(beta_inter) * 2000.0,
     "beta_premium + beta_interaccion x 2000: contrafactual completo a 2000 pie2"),
    ("efecto_premium_pendiente_2000_usd", float(beta_inter) * 2000.0,
     "el +85 400 publicado: SOLO la diferencia de pendientes a 2000 pie2 (~p80 del stock)"),
]:
    ws15.append(list(_fila))

# Hoja 16: modelos_drills - los modelos PARALELOS que usan los drills 1-3 (no son los del
# contrato: trio de garaje n=2924, interaccion sobre clean n=2925, comparacion A/B n=2923),
# registrados con sus cifras exactas para que la evaluacion tenga traza. Hoja NUEVA declarada.
_d1 = clean.dropna(subset=["SalePrice", "Gr Liv Area", "Garage Cars", "Garage Area"]).copy()
_X_d1 = sm.add_constant(_d1[["Gr Liv Area", "Garage Cars", "Garage Area"]])
_corr_d1 = _d1[["Gr Liv Area", "Garage Cars", "Garage Area"]].corr()
_vif_d1 = {c: float(variance_inflation_factor(_X_d1.values, i))
           for i, c in enumerate(_X_d1.columns)}
_d2 = clean.copy()
_d2["premium"] = _d2["Neighborhood"].isin(BARRIOS_PREMIUM).astype(int)
_d2["Area_x_premium"] = _d2["Gr Liv Area"] * _d2["premium"]
_m_d2 = sm.OLS(_d2["SalePrice"],
               sm.add_constant(_d2[["Gr Liv Area", "premium", "Area_x_premium"]])).fit()
_PA_drill = ["Gr Liv Area", "Overall Qual", "Year Built"]
_m_A = sm.OLS(lab["SalePrice"], sm.add_constant(lab[_PA_drill])).fit()
_rmse_A = -cross_val_score(LinearRegression(), lab[_PA_drill].values, y_lab, cv=kf,
                           scoring="neg_root_mean_squared_error", n_jobs=1)
_PM_drill = _PA_drill + ["Mo Sold"]
_m_M = sm.OLS(lab["SalePrice"], sm.add_constant(lab[_PM_drill])).fit()
_rmse_M = -cross_val_score(LinearRegression(), lab[_PM_drill].values, y_lab, cv=kf,
                           scoring="neg_root_mean_squared_error", n_jobs=1)
ws16 = wb.create_sheet("modelos_drills")
ws16.append(["modelo", "concepto", "valor", "nota"])
for _fila in [
    ("drill1_trio_garaje", "n", float(len(_d1)), "clean sin NaN de garaje: 2924 (no 2925)"),
    ("drill1_trio_garaje", "corr_grliv_garagecars",
     float(_corr_d1.loc["Gr Liv Area", "Garage Cars"]), ""),
    ("drill1_trio_garaje", "corr_grliv_garagearea",
     float(_corr_d1.loc["Gr Liv Area", "Garage Area"]), ""),
    ("drill1_trio_garaje", "corr_garagecars_garagearea",
     float(_corr_d1.loc["Garage Cars", "Garage Area"]), "el par ~0,892 que dispara el VIF"),
    ("drill1_trio_garaje", "r2aux_grlivarea", 1.0 - 1.0 / _vif_d1["Gr Liv Area"], ""),
    ("drill1_trio_garaje", "r2aux_garagecars", 1.0 - 1.0 / _vif_d1["Garage Cars"], ""),
    ("drill1_trio_garaje", "r2aux_garagearea", 1.0 - 1.0 / _vif_d1["Garage Area"], ""),
    ("drill1_trio_garaje", "vif_grlivarea", _vif_d1["Gr Liv Area"], ""),
    ("drill1_trio_garaje", "vif_garagecars", _vif_d1["Garage Cars"],
     "supera 5; Garage Area queda por debajo (NO 'ambos superan 5')"),
    ("drill1_trio_garaje", "vif_garagearea", _vif_d1["Garage Area"], ""),
    ("drill1_trio_garaje", "vif_const", _vif_d1["const"], "por construccion; no se interpreta"),
    ("drill2_interaccion_2925", "n", float(len(_d2)),
     "clean completo (2925): difiere del n=2923 de la hoja interaccion"),
    ("drill2_interaccion_2925", "n_premium", float(int(_d2["premium"].sum())), "286 casas premium"),
    ("drill2_interaccion_2925", "beta0_const", float(_m_d2.params["const"]), ""),
    ("drill2_interaccion_2925", "beta_grlivarea", float(_m_d2.params["Gr Liv Area"]), ""),
    ("drill2_interaccion_2925", "beta_premium", float(_m_d2.params["premium"]), ""),
    ("drill2_interaccion_2925", "p_premium", float(_m_d2.pvalues["premium"]),
     "no significativo (0,279)"),
    ("drill2_interaccion_2925", "beta_interaccion", float(_m_d2.params["Area_x_premium"]), ""),
    ("drill2_interaccion_2925", "r2", float(_m_d2.rsquared), ""),
    ("drill3_modeloA_2923", "r2_crudo", float(_m_A.rsquared),
     "0,775003 (con Mo Sold sube apenas a 0,775010)"),
    ("drill3_modeloA_2923", "r2_ajustado", float(_m_A.rsquared_adj), ""),
    ("drill3_modeloA_2923", "cv_rmse_medio", float(_rmse_A.mean()), ""),
    ("drill3_modeloA_2923", "cv_rmse_sd", float(_rmse_A.std()), ""),
    ("drill3_modeloA_mas_mosold", "r2_crudo", float(_m_M.rsquared), "el crudo casi no se mueve"),
    ("drill3_modeloA_mas_mosold", "r2_ajustado", float(_m_M.rsquared_adj), "el ajustado baja"),
    ("drill3_modeloA_mas_mosold", "cv_rmse_medio", float(_rmse_M.mean()), "la CV empeora"),
    ("drill3_modeloA_mas_mosold", "cv_rmse_sd", float(_rmse_M.std()), ""),
    ("drill3_modeloA_mas_mosold", "p_mosold", float(_m_M.pvalues["Mo Sold"]), "ruido (p=0,77)"),
]:
    ws16.append(list(_fila))

# Hoja 17: dummy_log (demo ejecutable de la P7 del control corto: la dummy premium en el modelo log)
ws17 = wb.create_sheet("dummy_log")
ws17.append(["concepto", "valor"])
_X_dl = sm.add_constant(lab[["Gr Liv Area", "Garage Cars", "Total Bsmt SF", "premium"]])
_m_dl = sm.OLS(np.log(lab["SalePrice"]), _X_dl).fit()
_b_dl = float(_m_dl.params["premium"])
ws17.append(["n", float(int(_m_dl.nobs))])
ws17.append(["beta_premium_log", _b_dl])
ws17.append(["efecto_pct_premium", float((np.exp(_b_dl) - 1) * 100)])
ws17.append(["p_valor_premium", float(_m_dl.pvalues["premium"])])
ws17.append(["lectura", "Con log(Y) la dummy se lee vs. la categoria base: "
                        "beta 0.1535 -> e^beta - 1 = +16.6 pct (mecanica de la P7)"])

wb.save(RUTA_XLSX)
print("Excel guardado en:", RUTA_XLSX)
print("Hojas:", wb.sheetnames)

**📖 Cómo se lee.** El Excel queda con las **6 hojas de contrato** más **11 auxiliares declaradas**. `regresion_ames` fija la réplica (corr 0,7068, 5 atípicas, R² simple 0,5176, R² múltiple log 0,8469); `diagnostico_vif`, `supuestos`, `cv_modelos`, `interaccion` y `coeficientes_multiple` (los coeficientes del modelo en log con su efecto %) guardan el resto; y las auxiliares añaden la inferencia robusta, la reconciliación de los cuatro R², el IC bootstrap, el panel de supuestos, la conversión a dólares y —en `cifras_titulares`— **toda cifra titular de la guía que antes no vivía en el contrato**; `retransformacion_duan` cierra la única comparación que faltaba: el RMSE del modelo **log** llevado a **USD** con el factor de smearing (**27 282 USD** en CV, frente a **33 308** del modelo en nivel). La subsección «Verificación desde la base» comprueba que estos valores son **producto de la ejecución**, no tecleados, y el material de referencia de la sesión los re-deriva uno a uno por una vía independiente.

### ✅ Verificación desde la base — transversal (subsección 6.1)

Antes de que el deck y la evaluación confíen en el Excel, se comprueba que ese registro es **producto de ejecutar el código sobre la base**, no un valor editado. Se **recomputan** desde los datos crudos ya cargados —de forma independiente de los modelos ya ajustados— el **R² múltiple (log)**, el **VIF de Overall Qual** y el **SE robusto HC3**, y se cruzan con las celdas del Excel mediante `assert`. Refleja, visible y explicada, la lógica de el material de referencia de la sesión.

**🔎 Qué hace este código.** (1) Recalcula el `R²` múltiple resolviendo `β = (XᵀX)⁻¹Xᵀy` sobre `log(SalePrice)`; (2) recalcula el VIF de `Overall Qual` como `1/(1−R²ⱼ)`; (3) reconstruye el SE robusto HC3 con la matriz sándwich. Lee las hojas del Excel y comprueba con `assert` que **recomputado ≈ paper ≈ Excel**. No escribe en el Excel.

In [ ]:
# Verificación desde la base: recomputar y cruzar con el Excel (NO escribe en el Excel)
from openpyxl import load_workbook

# (1) R² múltiple (log) desde las ecuaciones normales
Xv = np.column_stack([np.ones(len(mult)), mult[PREDS].to_numpy(float)])
yv = np.log(mult["SalePrice"].to_numpy(float))
beta_v = np.linalg.solve(Xv.T @ Xv, Xv.T @ yv)
r2_mult_v = 1 - np.sum((yv - Xv @ beta_v) ** 2) / np.sum((yv - yv.mean()) ** 2)

# (2) VIF de Overall Qual = 1/(1 - R²_j)
otros_v = [p for p in PREDS if p != "Overall Qual"]
r2_j_v = sm.OLS(mult["Overall Qual"], sm.add_constant(mult[otros_v])).fit().rsquared
vif_oq_v = 1.0 / (1.0 - r2_j_v)

# (3) SE robusto HC3 del coeficiente de Gr Liv Area (modelo simple en nivel), matriz sándwich
Xh = np.column_stack([np.ones(len(clean)), clean["Gr Liv Area"].to_numpy(float)])
yh = clean["SalePrice"].to_numpy(float)
bh = np.linalg.solve(Xh.T @ Xh, Xh.T @ yh)
eh = yh - Xh @ bh
XtXinv = np.linalg.inv(Xh.T @ Xh)
hh = np.sum((Xh @ XtXinv) * Xh, axis=1)                 # leverage h_i
meat = Xh.T @ (Xh * ((eh / (1 - hh)) ** 2)[:, None])    # Σ e_i²/(1-h_i)² x_i x_iᵀ
se_hc3_v = np.sqrt(np.diag(XtXinv @ meat @ XtXinv))[1]

wbv = load_workbook(RUTA_XLSX, data_only=True)
reg = {wbv["regresion_ames"][f"A{r}"].value: wbv["regresion_ames"][f"B{r}"].value for r in range(2, 6)}
vifx = {wbv["diagnostico_vif"][f"A{r}"].value: wbv["diagnostico_vif"][f"B{r}"].value for r in range(2, 7)}
sup = {(wbv["supuestos"][f"A{r}"].value, wbv["supuestos"][f"B{r}"].value): wbv["supuestos"][f"C{r}"].value
       for r in range(2, 7)}

tabla = pd.DataFrame({
    "recomputado (venv)": [round(r2_mult_v, 4), round(vif_oq_v, 4), round(se_hc3_v, 4)],
    "paper / referencia": [0.847, 2.47, 3.01],
    "Excel (contrato)":   [round(reg["r2_multiple_log"], 4), round(vifx["Overall Qual"], 4),
                           round(sup[("SE_grlivarea", "robusto_HC3")], 4)],
}, index=["R2 multiple (log)", "VIF Overall Qual", "SE HC3 Gr Liv Area"])
print(tabla.to_string())

assert abs(r2_mult_v - reg["r2_multiple_log"]) < 1e-6
assert abs(vif_oq_v - vifx["Overall Qual"]) < 1e-6
assert abs(se_hc3_v - sup[("SE_grlivarea", "robusto_HC3")]) < 1e-6
assert abs(r2_mult_v - 0.847) <= 0.01 and abs(vif_oq_v - 2.47) <= 0.05   # dentro de tolerancia del paper
print("\nOK: recomputado desde la base ~ paper (tolerancia) y ~ Excel (contrato).")

**📖 Cómo se lee.** Las tres columnas coinciden: lo **recomputado** desde los datos reproduce el **paper/target** (R² 0,847; VIF 2,47; SE HC3 3,01) y coincide con el **Excel** hasta el sexto decimal. Los `assert` fallarían si alguien editara el Excel manualmente o si la base cambiara: por eso el registro es *auditable*. La versión ejecutable con carga de la base vive en el material de referencia de la sesión.

### Figuras de resultados (leídas del Excel) — transversal (subsección 6.2)

**🔎 Qué hace este código.** Lee las hojas `diagnostico_vif` y `cv_modelos` del Excel recién escrito y genera dos figuras de resultados: el **VIF por predictor** (con el umbral 5) y la **comparación de modelos por CV 5-fold** (R² y RMSE) — todo con valores **leídos del Excel**, no de objetos en memoria.

In [ ]:
# Figuras de RESULTADOS: se generan LEYENDO el Excel
vif_xl = pd.read_excel(RUTA_XLSX, sheet_name="diagnostico_vif")
cv_xl = pd.read_excel(RUTA_XLSX, sheet_name="cv_modelos")

# (1) VIF por predictor
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(vif_xl["predictor"], vif_xl["VIF"], color=UPC_ROJO)
ax.axvline(5, color=UPC_TINTA, ls="--", lw=1.2, label="Umbral VIF = 5 (atención)")
ax.set_xlabel("Factor de inflación de la varianza (VIF)")
ax.set_title("Diagnóstico de multicolinealidad — VIF por predictor")
ax.legend(loc="lower right")
for y, v in enumerate(vif_xl["VIF"]):
    ax.text(v + 0.05, y, "%.2f" % v, va="center", fontsize=9)
mostrar(fig, os.path.join(FIG_DIR, "S04_vif_predictores.png"))

# (2) Comparación de modelos por CV 5-fold
fig, (axr, axe) = plt.subplots(1, 2, figsize=(11, 4.5))
etq = ["Simple\n(Gr Liv Area)", "Múltiple\n(5 predictores)"]
axr.bar(etq, cv_xl["cv_r2_medio"], yerr=cv_xl["cv_r2_sd"], capsize=6, color=[UPC_ROSA, UPC_ROJO])
axr.set_ylabel("R² medio (CV 5-fold)"); axr.set_title("Poder explicativo fuera de muestra (R²)"); axr.set_ylim(0, 1)
for i, v in enumerate(cv_xl["cv_r2_medio"]):
    axr.text(i, v + 0.03, "%.3f" % v, ha="center", fontweight="bold")
axe.bar(etq, cv_xl["cv_rmse_medio"], yerr=cv_xl["cv_rmse_sd"], capsize=6, color=[UPC_ROSA, UPC_ROJO])
axe.set_ylabel("RMSE medio (USD, CV 5-fold)"); axe.set_title("Error de predicción fuera de muestra (RMSE)")
for i, v in enumerate(cv_xl["cv_rmse_medio"]):
    axe.text(i, v + 800, "%.0f" % v, ha="center", fontweight="bold")
fig.suptitle("Comparación de modelos por validación cruzada — el múltiple generaliza mejor", fontsize=12)
mostrar(fig, os.path.join(FIG_DIR, "S04_comparacion_cv.png"))

**📖 Cómo se lee.** El VIF de los cinco predictores queda a la izquierda del umbral 5 ⇒ coeficientes interpretables. La comparación por CV muestra el salto de poder explicativo (R² 0,52 → 0,82) y la caída del error (RMSE 54 500 → 33 300 USD). Cualquier cambio en el contrato del Excel se reflejaría automáticamente en estas figuras.

## 4.3 en profundidad — Construcción de la regresión múltiple desde cero (Sección 7 del cuaderno)

El alumno **rearma el flujo completo sin los helpers ni los objetos ya calculados**: del **CSV crudo** a la matriz de diseño (numérica y **con dummies**), el ajuste por ecuaciones normales, el VIF, el SE robusto y la CV. El objetivo es reproducir, de forma independiente, el **mismo contrato** del Excel. Si coincide (con `assert`), queda demostrado que **es el método —no un atajo— el que produce la réplica**.

**🔎 Qué hace este código.** Lee `AmesHousing.csv` desde cero, remueve las 5 atípicas, arma la matriz de diseño `X = [1, 5 predictores]`, resuelve `β = (XᵀX)⁻¹Xᵀy` sobre `log(SalePrice)` con álgebra lineal y calcula `R²` y `R²` ajustado. Comprueba con `assert` que reproduce la hoja `regresion_ames` del Excel.

In [ ]:
# (a) CSV crudo -> matriz de diseño numérica -> β = (XᵀX)⁻¹Xᵀy sobre log(precio)
csv_ames = os.path.join(DATA_DIR, "AmesHousing.csv")
raw = pd.read_csv(csv_ames) if os.path.exists(csv_ames) else ames.copy()
clean_z = raw[raw["Gr Liv Area"] <= 4000].dropna(subset=PREDS + ["SalePrice"]).copy()

Xz = np.column_stack([np.ones(len(clean_z)), clean_z[PREDS].to_numpy(float)])
yz = np.log(clean_z["SalePrice"].to_numpy(float))
beta_z = np.linalg.solve(Xz.T @ Xz, Xz.T @ yz)          # β = (XᵀX)⁻¹Xᵀy
yhat_z = Xz @ beta_z
n_z, p_z = len(clean_z), len(PREDS)
SSE_z = np.sum((yz - yhat_z) ** 2); SST_z = np.sum((yz - yz.mean()) ** 2)
r2_z = 1 - SSE_z / SST_z
r2_adj_z = 1 - (1 - r2_z) * (n_z - 1) / (n_z - p_z - 1)

print("n =", n_z, "| predictores =", p_z)
print("R² múltiple (log) desde cero = %.6f  (contrato 0.846943)" % r2_z)
print("R² ajustado                   = %.6f" % r2_adj_z)

cw = load_workbook(RUTA_XLSX, data_only=True)
r2_contrato = cw["regresion_ames"]["B5"].value
assert abs(r2_z - r2_contrato) < 1e-6
print("\nOK: la matriz de diseño desde el CSV reproduce el R² del contrato.")

**📖 Cómo se lee.** Partiendo del CSV crudo y sin reutilizar ningún modelo, el álgebra matricial reproduce el `R² = 0,846943` del contrato. La réplica no dependía de un estado oculto del cuaderno: el **método** la produce.

**🔎 Qué hace este código.** Sobre la misma base cruda, reconstruye de forma manual el **VIF de los 5 predictores** (`1/(1−R²ⱼ)`) y el **SE robusto HC3** del modelo simple en nivel (matriz sándwich), y comprueba con `assert` que reproducen las hojas `diagnostico_vif` y `supuestos` del Excel.

In [ ]:
# (b) VIF de los 5 a mano + SE robusto HC3 a mano, contra el contrato
vif_desde_cero = {}
for j in PREDS:
    resto = [p for p in PREDS if p != j]
    r2j = sm.OLS(clean_z[j], sm.add_constant(clean_z[resto])).fit().rsquared
    vif_desde_cero[j] = 1.0 / (1.0 - r2j)
print("VIF desde cero:")
for j in PREDS:
    print("  %-14s %.4f" % (j, vif_desde_cero[j]))

# SE robusto HC3 del modelo simple en nivel (set limpio completo, 2925 filas, como el contrato)
clean_lvl = raw[raw["Gr Liv Area"] <= 4000]
Xn = np.column_stack([np.ones(len(clean_lvl)), clean_lvl["Gr Liv Area"].to_numpy(float)])
yn = clean_lvl["SalePrice"].to_numpy(float)
bn = np.linalg.solve(Xn.T @ Xn, Xn.T @ yn)
en = yn - Xn @ bn
XtXi = np.linalg.inv(Xn.T @ Xn)
hn = np.sum((Xn @ XtXi) * Xn, axis=1)
se_hc3_z = np.sqrt(np.diag(XtXi @ (Xn.T @ (Xn * ((en / (1 - hn)) ** 2)[:, None])) @ XtXi))[1]
print("\nSE robusto HC3 (Gr Liv Area) desde cero = %.4f  (contrato 3.0075)" % se_hc3_z)

vif_x = {cw["diagnostico_vif"][f"A{r}"].value: cw["diagnostico_vif"][f"B{r}"].value for r in range(2, 7)}
se_hc3_contrato = cw["supuestos"]["C6"].value
assert abs(vif_desde_cero["Overall Qual"] - vif_x["Overall Qual"]) < 1e-6
assert abs(se_hc3_z - se_hc3_contrato) < 1e-6
print("OK: VIF y SE robusto desde cero reproducen el contrato.")

**📖 Cómo se lee.** El VIF de cada predictor y el SE robusto HC3 se reconstruyen de forma manual y coinciden con el Excel: ni el diagnóstico de colinealidad ni la corrección de la inferencia son cajas negras.

**🔎 Qué hace este código.** Cierra el flujo: arma la **matriz de diseño con dummies** (superficie, `premium` y su interacción) y resuelve `β = (XᵀX)⁻¹Xᵀy` para reproducir la hoja `interaccion`; y ejecuta una **CV 5-fold** (semilla 42) del modelo de 5 predictores para reproducir la hoja `cv_modelos`. Todo verificado con `assert`.

In [ ]:
# (c) Matriz de diseño CON dummies + interacción, y CV 5-fold desde cero
lab_z = raw[raw["Gr Liv Area"] <= 4000].dropna(
    subset=["SalePrice", "Gr Liv Area", "Neighborhood", "Overall Qual",
            "Garage Cars", "Total Bsmt SF", "Year Built"]).copy()
lab_z["premium"] = lab_z["Neighborhood"].isin(["NridgHt", "NoRidge", "StoneBr"]).astype(int)

# matriz de diseño con dummy: [1, área, premium, área×premium]
Xd = np.column_stack([np.ones(len(lab_z)), lab_z["Gr Liv Area"].to_numpy(float),
                      lab_z["premium"].to_numpy(float),
                      (lab_z["Gr Liv Area"] * lab_z["premium"]).to_numpy(float)])
yd = lab_z["SalePrice"].to_numpy(float)
beta_d = np.linalg.solve(Xd.T @ Xd, Xd.T @ yd)          # [b0, base, premium, interacción]
print("Pendiente base desde cero     = %.4f  (contrato 88.0495)" % beta_d[1])
print("Coef. interacción desde cero  = %.4f  (contrato 42.7024)" % beta_d[3])

# CV 5-fold del modelo múltiple (nivel), semilla 42
kf_z = KFold(n_splits=5, shuffle=True, random_state=42)
r2cv_z = cross_val_score(LinearRegression(), lab_z[PREDS].values, yd, cv=kf_z, scoring="r2").mean()
print("CV R² múltiple desde cero     = %.4f  (contrato 0.8194)" % r2cv_z)

intx = {cw["interaccion"][f"A{r}"].value: cw["interaccion"][f"B{r}"].value for r in range(2, 7)}
cv_mult = cw["cv_modelos"]["B3"].value
assert abs(beta_d[1] - intx["pendiente_base_no_premium"]) < 1e-4
assert abs(beta_d[3] - intx["coef_interaccion_area_x_premium"]) < 1e-4
assert abs(r2cv_z - cv_mult) < 1e-4
print("\nOK: interacción (con dummies) y CV desde cero reproducen el contrato.")

**📖 Cómo se lee.** La matriz de diseño **con la dummy y su interacción** reproduce la pendiente base (88,05) y el extra premium (+42,70), y la CV 5-fold reproduce el R² del modelo múltiple (0,8194). El pipeline completo —del CSV a la CV— es reproducible sin helpers.

**✍️ Ahora, por cuenta propia.** Se propone (a) añadir `Garage Area` junto a `Garage Cars` a la matriz de diseño y observar cómo **suben sus VIF** hacia 5; (b) reajustar el modelo con interacción usando otro conjunto de barrios «premium» y comparar el coeficiente de la interacción; y (c) recalcular la CV con `random_state` distinto y discutir por qué la media apenas cambia pero la sd sí.

## 4.6 — ¿Cuándo se puede confiar en un modelo con múltiples variables? Supuestos: cómo identificarlos y corregirlos (Sección 8 del cuaderno)

> **Fuente canónica:** la guía de supuestos de la sesión (qué es, cómo se identifica, cómo se corrige por método, alcance). Aquí se ejecutan los **diagnósticos**; su desarrollo teórico y sus fuentes viven en ese documento. **Ninguna celda de esta sección escribe en el Excel de contrato.**
>
> **Regla de alcance de S04.** El trabajo sobre cada supuesto llega hasta **(a) diagnosticarlo** con su prueba/gráfico y un umbral, y **(b) aplicar la corrección clásica del OLS múltiple**: **transformar** (`log`/Box-Cox), **corregir la inferencia** con errores robustos **HC3**, y **decidir predictores por diagnóstico** (quitar/combinar colineales). La regularización (Ridge/Lasso, S05), el PCA (S06), las series (S11) y la causalidad (S12) se **nombran**, no se ejecutan.

**Supuestos del OLS múltiple (para que sea BLUE)** — `SUPUESTOS_S04.md`, Parte 1:

| Supuesto (Wooldridge) | Cómo identificar (umbral) | Cómo corregir (método) | Alcance |
|---|---|---|---|
| **1.1 Linealidad** (MLR.1) | Residuales vs. ajustados (curva/arco) | Transformar `X`/`Y` (log); interacciones — *splines → S05* | Diagnóstico + transformar **S04** |
| **1.2 Exogeneidad `E[ε\|X]=0`** (MLR.4) | Razonar el dominio (no hay test); coeficiente que cambia al añadir otro | Incluir controles/dummies — *causalidad → S12* | Controlar + advertir **S04** |
| **1.3 Homocedasticidad** (MLR.5) | Residuales (abanico); **Breusch-Pagan** (`p<0,05`) | **Errores robustos HC3**; transformar `Y` (log) | Diagnóstico + HC3/log **S04** |
| **1.4 No autocorrelación** (MLR.2) | **Durbin-Watson** (≈2) **solo si hay orden** | Newey-West/HAC — *series → S11* | Detectar/reportar **S04** |
| **1.5 Sin multicolinealidad** (MLR.3) — **ESTRELLA** | **VIF** (`>5`/`>10`); matriz de correlaciones (`>0,8`) | **Quitar/combinar** predictores — *Ridge → S05, PCA → S06* | Diagnóstico + quitar/combinar **S04** |

**Inferencia, diagnóstico de datos y decisiones de modelado** — `SUPUESTOS_S04.md`, Partes 2 y 3:

| Punto | Cómo identificar (umbral) | Cómo corregir / especificar | Alcance |
|---|---|---|---|
| **2.1 Normalidad de los errores** (MLR.6; NO Gauss-Markov) | Q-Q (colas en S); Omnibus/Jarque-Bera (`p<0,05`) | Transformar `Y`; **TCL** con n grande | **S04** (leer Q-Q; TCL) |
| **2.2 Influyentes / leverage / atípicos** | **Distancia de Cook** (`>1` claro, `>4/n` candidatos); leverage; estudentizados | Investigar; sensibilidad; **remoción justificada** | Diagnóstico + remoción justificada **S04** |
| **3.1 Dummies / dummy redundante** | Colinealidad perfecta (VIF ∞, coef `NaN`) | **`drop_first=True`** (categoría de referencia) | **S04** (enlaza con 1.5) |
| **3.2 Interacciones (efecto condicional)** | p-valor del término `X₁·X₂`; pendientes por grupo | Incluir con **principio de jerarquía**; leer `β₁+β₃X₂` | **S04 — núcleo del laboratorio** |
| **3.3 Transformación log de la respuesta** | Asimetría de `Y`; embudo + Q-Q con colas; Box-Cox `λ→0` | Ajustar sobre `log(Y)`; leer en % | **S04** (transformar) |

**🔎 Qué hace este código.** Diagnostica la **multicolinealidad** (supuesto 1.5, la estrella): imprime la tabla de **VIF** de los 5 predictores del modelo múltiple y traza el **mapa de calor de correlaciones** entre las variables numéricas clave — el diagnóstico rápido previo al VIF.

In [ ]:
# Supuesto 1.5 (multicolinealidad): tabla VIF + matriz de correlaciones (NO escribe en el Excel)
Xvif8 = sm.add_constant(mult[PREDS])
vif_tabla = pd.DataFrame({"predictor": PREDS,
                          "VIF": [variance_inflation_factor(Xvif8.values, i + 1) for i in range(len(PREDS))]})
print(vif_tabla.round(3).to_string(index=False))
print("\nTodos los VIF < 5 -> sin multicolinealidad preocupante (máx Overall Qual ≈ 2.47).")

cols_num = ["SalePrice", "Gr Liv Area", "Overall Qual", "Garage Cars",
            "Total Bsmt SF", "Year Built", "Garage Area", "Full Bath"]
corr_mat = ames[cols_num].corr()
fig, ax = plt.subplots(figsize=(8.5, 7))
sns.heatmap(corr_mat, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, cbar_kws={"label": "Correlación de Pearson"}, ax=ax)
ax.set_title("Correlación entre variables numéricas — Ames Housing")
mostrar(fig, os.path.join(FIG_DIR, "S04_eda_heatmap_correlacion.png"))

**📖 Cómo se lee.** Los cinco VIF quedan entre 1,50 y 2,47, **todos < 5** ⇒ los coeficientes son interpretables con confianza. En el mapa de calor, el par `Garage Cars`–`Garage Area` (~0,89) es el más alto: por eso incluir **ambos** eleva su VIF hacia 5 (demo 3.2). **Corrección (identificar → corregir):** ante VIF alto, **quitar o combinar** el predictor redundante; la penalización (Ridge) es S05 y el PCA es S06. Método en `SUPUESTOS_S04.md`, Parte 1.5.

**🔎 Qué hace este código.** Diagnostica la **homocedasticidad** (supuesto 1.3): ejecuta **Breusch-Pagan** en el modelo en nivel, grafica sus **residuales vs. ajustados** (abanico) y compara el SE clásico con el **robusto HC3** — la corrección de la sesión.

In [ ]:
# Supuesto 1.3 (homocedasticidad): Breusch-Pagan + residuales (abanico) + HC3 (NO escribe en el Excel)
m_nivel = sm.OLS(clean["SalePrice"], sm.add_constant(clean[["Gr Liv Area"]])).fit()
m_hc3 = sm.OLS(clean["SalePrice"], sm.add_constant(clean[["Gr Liv Area"]])).fit(cov_type="HC3")
bp8 = het_breuschpagan(m_nivel.resid, m_nivel.model.exog)
print("Breusch-Pagan (nivel): LM = %.1f, p = %.2e  -> heterocedasticidad severa" % (bp8[0], bp8[1]))
print("SE Gr Liv Area  clásico = %.3f   robusto HC3 = %.3f  (+%.0f%%)"
      % (m_nivel.bse["Gr Liv Area"], m_hc3.bse["Gr Liv Area"],
         100 * (m_hc3.bse["Gr Liv Area"] / m_nivel.bse["Gr Liv Area"] - 1)))

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.scatter(m_nivel.fittedvalues, m_nivel.resid, alpha=0.3, color=UPC_ROJO, edgecolor="none", s=14)
ax.axhline(0, color=UPC_TINTA, lw=1.2, ls="--")
ax.set_xlabel("Valores ajustados (USD)"); ax.set_ylabel("Residuales")
ax.set_title("Ames en nivel: residuales vs. ajustados (abanico = heterocedasticidad)")
mostrar(fig, os.path.join(FIG_DIR, "S04_eda_residuales_ajustados.png"))

**📖 Cómo se lee.** Los residuales **abren un abanico** (la dispersión crece con el precio ajustado) y Breusch-Pagan lo confirma (`p ≈ 0`). **⚠️** La heterocedasticidad **no sesga** los β, pero invalida sus errores estándar. **Corrección (identificar → corregir):** reportar **errores robustos HC3** (el SE sube de 2,08 a 3,01, +45 %) y/o **transformar** a `log(precio)` — las dos palancas de S04. Método en `SUPUESTOS_S04.md`, Parte 1.3.

**🔎 Qué hace este código.** Diagnostica la **no autocorrelación / independencia** (supuesto 1.4): calcula el **Durbin-Watson** del modelo múltiple y ejecuta el **experimento del orden** (original, barajado con semilla 42, ordenado por barrio) que demuestra que el orden del CSV **es informativo** —viene agrupado geográficamente por PID—, de modo que el DW **sí se interpreta** aquí: señal de **dependencia espacial residual**. Cierra comparando el SE clásico, el HC3 y el **SE agrupado por barrio (cluster)** del modelo desplegado.

In [ ]:
# Supuesto 1.4 (no autocorrelación / independencia): Durbin-Watson + experimento del orden
# (NO escribe en el Excel; la hoja `se_cluster_barrio` la escribe la celda de exportación)
dw = durbin_watson(modelo_multiple.resid)
res_dw = modelo_multiple.resid.to_numpy(float)
barrio_dw = mult["Neighborhood"].to_numpy()
dw_de = lambda r: float(np.sum(np.diff(r) ** 2) / np.sum(r ** 2))
dw_baraja = dw_de(res_dw[np.random.default_rng(42).permutation(len(res_dw))])
dw_barrio = dw_de(res_dw[np.argsort(barrio_dw, kind="quicksort")])
contig = (ames["Neighborhood"].values[1:] == ames["Neighborhood"].values[:-1]).mean() * 100

print("Durbin-Watson (modelo múltiple) = %.3f  ->  rho ~ %.2f" % (dw, 1 - dw / 2))
print("\nExperimento del orden (¿el orden de las filas significa algo?):")
print("  original (orden PID del CSV)       : DW = %.3f" % dw)
print("  barajado (semilla 42)              : DW = %.3f  (sin orden, vuelve a ~2)" % dw_baraja)
print("  ordenado por barrio (Neighborhood) : DW = %.3f  (reproduce el desvío)" % dw_barrio)
print("  contigüidad de barrio en el CSV    : %.1f %% de filas contiguas (azar ~6,7 %%)" % contig)
print("VEREDICTO: el orden del CSV ES informativo (agrupado geográficamente por PID), así que")
print("DW = %.2f SÍ se interpreta: dependencia espacial residual (rho ~ %.2f intra-barrio)."
      % (dw, 1 - dw / 2))

# Consecuencia sobre la inferencia: SE clásico vs HC3 vs agrupado por barrio (cluster)
modelo_multiple_cl = sm.OLS(y_log, Xm).fit(cov_type="cluster",
                                           cov_kwds={"groups": mult["Neighborhood"]})
comp_se = pd.DataFrame({
    "SE_clasico": modelo_multiple.bse,
    "SE_HC3": modelo_multiple_hc3.bse,
    "SE_cluster_barrio": modelo_multiple_cl.bse,
})
comp_se["cluster/clasico"] = comp_se["SE_cluster_barrio"] / comp_se["SE_clasico"]
print("\nSE clásico vs HC3 vs agrupado por barrio (%d clusters; los coeficientes NO cambian):"
      % mult["Neighborhood"].nunique())
print(comp_se.round(6).to_string())
print("\nLECTURA: la dependencia intra-barrio infla la incertidumbre honesta hasta ~2-3x el SE")
print("clásico; HC3 corrige heterocedasticidad, NO dependencia. El tratamiento espacial completo")
print("(modelos espaciales, GroupKFold por barrio) es materia de S11+ y cursos posteriores.")

**📖 Cómo se lee.** `DW ≈ 1,68` — y aquí la lectura correcta es **«señal de dependencia espacial»**. ⚠️ **Criterio canónico (corregido contra los datos):** el Durbin-Watson compara cada residual con el **anterior**, de modo que solo significa algo cuando las filas están ordenadas por algo real (tiempo o espacio). El experimento del orden muestra que en Ames **sí lo están**: el CSV viene **agrupado geográficamente** (orden por PID; ~86 % de filas contiguas comparten barrio), barajarlo devuelve el DW a ≈ 1,97 (≈ 2) y **reordenar por barrio reproduce ≈ 1,70 ≈ 1,68** — el desvío no es ruido de archivo sino **correlación positiva de los residuales dentro de cada barrio** (`ρ̂ ≈ 1 − DW/2 ≈ 0,16`, que con `n ≈ 2900` rechaza el nulo). **Consecuencias honestas:** (a) los SE clásicos **y** los HC3 son **optimistas** (HC3 corrige heterocedasticidad, no dependencia); (b) el bootstrap i.i.d. de casos hereda el mismo supuesto de independencia; (c) la CV `KFold` aleatoria mezcla barrios entre folds (fuga espacial leve). **El remedio de S04** es reportar los **SE agrupados por barrio** (hoja `se_cluster_barrio`: el SE de `Overall Qual` pasa de 0,0033 clásico / 0,0047 HC3 a **0,0075 cluster**, y el IC del efecto se ensancha a [9,3 %; 12,5 %]); el tratamiento completo (modelos espaciales, GroupKFold por barrio, Newey-West para series) es **S11+**. Método en `SUPUESTOS_S04.md`, Parte 1.4.

**🔎 Qué hace este código.** Diagnostica la **normalidad** de los residuales (supuesto 2.1) y el efecto de la **transformación log** (3.3): traza el **Q-Q plot** del modelo múltiple (log) y los **histogramas** de `SalePrice` vs. `log(SalePrice)`.

In [ ]:
# Supuestos 2.1 (normalidad) y 3.3 (log): Q-Q plot + histogramas (NO escribe en el Excel)
fig = sm.qqplot(modelo_multiple.resid, line="45", fit=True, markersize=4, alpha=0.4)
fig.set_size_inches(6.5, 6)
fig.axes[0].set_title("Q-Q plot de los residuales — modelo múltiple log(SalePrice)")
fig.axes[0].set_xlabel("Cuantiles teóricos (normal)"); fig.axes[0].set_ylabel("Cuantiles de los residuales")
mostrar(fig, os.path.join(FIG_DIR, "S04_eda_qqplot.png"))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.5))
a1.hist(ames["SalePrice"], bins=40, color=UPC_ROSA, edgecolor="white")
a1.set_title("SalePrice (asimétrica a la derecha)"); a1.set_xlabel("Precio (USD)"); a1.set_ylabel("Frecuencia")
a2.hist(np.log(ames["SalePrice"]), bins=40, color=UPC_ROJO, edgecolor="white")
a2.set_title("log(SalePrice) (aproxima la simetría)"); a2.set_xlabel("log(precio)"); a2.set_ylabel("Frecuencia")
fig.suptitle("La transformación logarítmica estabiliza la distribución del precio", fontsize=12)
mostrar(fig, os.path.join(FIG_DIR, "S04_eda_hist_log.png"))

**📖 Cómo se lee.** Los puntos del Q-Q caen casi sobre la diagonal con colas levemente despegadas; con `n ≈ 2900` el **TCL** vuelve la inferencia aproximadamente válida (la normalidad no hace falta para BLUE). El histograma muestra por qué se transforma: `SalePrice` es asimétrica a la derecha y `log(SalePrice)` **aproxima la simetría**. **Corrección:** `log(Y)` ataca a la vez asimetría (2.1) y embudo (1.3). Método en `SUPUESTOS_S04.md`, Partes 2.1 y 3.3.

**🔎 Qué hace este código.** Diagnostica **observaciones influyentes** (supuesto 2.2): calcula la **distancia de Cook**, el **leverage** y los **residuos estudentizados** del modelo múltiple, cuenta cuántos superan sus umbrales y traza la dispersión `SalePrice` vs. `Gr Liv Area` resaltando las **5 atípicas de De Cock**.

In [ ]:
# Supuesto 2.2 (influyentes): Cook + leverage + estudentizados + dispersión de las 5 (NO escribe en el Excel)
infl8 = modelo_multiple.get_influence()
cook8 = infl8.cooks_distance[0]
leverage8 = infl8.hat_matrix_diag
stud8 = infl8.resid_studentized_internal
n8 = len(mult); p8 = len(PREDS) + 1
print("n = %d | 4/n = %.5f" % (n8, 4 / n8))
print("Cook > 4/n (candidatos)     : %d   |  Cook máximo = %.4f (regla clásica Cook > 1)" % ((cook8 > 4 / n8).sum(), cook8.max()))
print("leverage > 2p/n             : %d" % (leverage8 > 2 * p8 / n8).sum())
print("residuos |estudentizado| > 3: %d" % (np.abs(stud8) > 3).sum())

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(ames["Gr Liv Area"], ames["SalePrice"], s=12, alpha=0.30, color=UPC_GRIS, label="Ventas (n=2930)")
ax.scatter(atipicas["Gr Liv Area"], atipicas["SalePrice"], s=90, color=UPC_ROJO, edgecolor="black",
           zorder=5, label="Atípicas > 4000 pie² (remover)")
ax.axvline(4000, color=UPC_ROJO, ls="--", lw=1, alpha=0.7)
ax.set_xlabel("Área habitable — Gr Liv Area (pie²)"); ax.set_ylabel("Precio de venta — SalePrice (USD)")
ax.set_title("Relación precio–superficie y las 5 ventas atípicas de De Cock")
ax.legend()
mostrar(fig, os.path.join(FIG_DIR, "S04_eda_dispersion_atipicas.png"))

**📖 Cómo se lee.** El **Cook máximo** queda muy por debajo de 1: aunque ~192 puntos superen el umbral sensible `4/n`, **ninguno determina por sí solo** la recta. **💡** El umbral `4/n` lista **candidatos a inspección**, no a borrado. La dispersión ubica las **5 casas > 4000 pie²** que De Cock recomienda remover — remoción **justificada por el dominio** (3 `Partial` + 1 `Abnorml` + 1 `Normal`), no borrado ciego. Método en `SUPUESTOS_S04.md`, Parte 2.2.

**🔎 Qué hace este código.** Demuestra el **error de la dummy redundante** (supuesto 3.1): codifica `Bldg Type` con **todas** las categorías (sin omitir la de referencia) **más** el intercepto, y muestra el VIF divergente — la colinealidad perfecta que impide estimar. Luego muestra la codificación correcta con `drop_first=True`.

In [ ]:
# Supuesto 3.1 (trampa de la dummy): colinealidad perfecta vs. drop_first (NO escribe en el Excel)
bt = ames.dropna(subset=["Bldg Type", "SalePrice"]).copy()

# INCORRECTO: todas las categorías + intercepto -> colinealidad perfecta
D_full = pd.get_dummies(bt["Bldg Type"], prefix="Bldg", drop_first=False, dtype=int)
X_trap = sm.add_constant(D_full.astype(float))
vif_trap = [variance_inflation_factor(X_trap.values, i) for i in range(1, X_trap.shape[1])]
print("Trampa (todas las dummies + const): VIF máx = %.1f  -> colinealidad perfecta (las dummies suman 1)"
      % max(vif_trap))

# CORRECTO: drop_first=True (categoría de referencia)
D_ok = pd.get_dummies(bt["Bldg Type"], prefix="Bldg", drop_first=True, dtype=int)
X_ok = sm.add_constant(D_ok.astype(float))
vif_ok = [variance_inflation_factor(X_ok.values, i) for i in range(1, X_ok.shape[1])]
print("Con drop_first=True               : VIF máx = %.2f  -> estimable; cada β = diferencia vs. base"
      % max(vif_ok))

**📖 Cómo se lee.** Con **todas** las dummies más el intercepto, el VIF tiende a infinito (las columnas suman 1 y replican la constante): el modelo **no se puede estimar**. Con `drop_first=True` el VIF vuelve a valores normales y cada coeficiente se lee como **diferencia respecto a la categoría base**. La dummy redundante es un caso de **colinealidad perfecta** (enlaza con 1.5). Método en `SUPUESTOS_S04.md`, Parte 3.1.

**🔎 Qué hace este código.** Diagnostica el **efecto condicional** de la **interacción** (supuesto 3.2): grafica las dos rectas `SalePrice ~ Gr Liv Area` —barrio no premium vs. premium— con sus pendientes distintas, la señal visual de que el efecto del tamaño **depende de la zona**.

In [ ]:
# Supuesto 3.2 (interacción): pendientes por grupo (NO escribe en el Excel)
b0_i = modelo_interaccion.params["const"]
b_prem = modelo_interaccion.params["premium"]
xs = np.linspace(lab["Gr Liv Area"].min(), lab["Gr Liv Area"].max(), 100)
recta_base = b0_i + beta_base * xs
recta_prem = (b0_i + b_prem) + (beta_base + beta_inter) * xs

fig, ax = plt.subplots(figsize=(8, 5.5))
m0 = lab["premium"] == 0
ax.scatter(lab.loc[m0, "Gr Liv Area"], lab.loc[m0, "SalePrice"], s=10, alpha=0.25, color=UPC_GRIS, label="No premium")
ax.scatter(lab.loc[~m0, "Gr Liv Area"], lab.loc[~m0, "SalePrice"], s=16, alpha=0.5, color=UPC_ROJO, label="Premium")
ax.plot(xs, recta_base, color=UPC_TINTA, lw=2, label="Pendiente base ≈ %.0f USD/pie²" % beta_base)
ax.plot(xs, recta_prem, color=UPC_ROJO, lw=2, ls="--", label="Pendiente premium ≈ %.0f USD/pie²" % (beta_base + beta_inter))
ax.set_xlabel("Gr Liv Area (pie²)"); ax.set_ylabel("SalePrice (USD)")
ax.set_title("Interacción zona × tamaño: la pendiente del precio depende del barrio")
ax.legend(fontsize=9)
mostrar(fig, os.path.join(FIG_DIR, "S04_interaccion_pendientes.png"))

**📖 Cómo se lee.** Las dos rectas **no son paralelas**: en barrio premium la pendiente (~131 USD/pie²) es más empinada que en el resto (~88 USD/pie²). Esa diferencia de pendientes **es** el coeficiente de la interacción (+42,7, `p ≈ 0`). **Corrección (especificar bien):** incluir la interacción con sus términos principales (jerarquía) y leer la pendiente condicionada `β₁ + β₃·premium`, no `β₁` aislada. Método en `SUPUESTOS_S04.md`, Parte 3.2.

### Panel de incertidumbre verificado — IC bootstrap y supuestos — capítulo 4.7 (subsección 8.9)

> **Fuentes:** la guía de supuestos de la sesión (supuestos) e la guía de interpretación de resultados «Sección 7-8» (inferencia robusta). Esta subsección **lee del Excel** las dos hojas auxiliares que la celda de exportación construyó —`bootstrap_ic` y `supuestos_panel`— y las muestra con su lectura honesta. **No escribe en el Excel.**

Dos preguntas de honestidad estadística cierran los diagnósticos:
1. **¿Cuánta incertidumbre carga la cifra insignia?** Un **intervalo de confianza bootstrap** del 95 % —remuestreo de casos, **sin suponer normalidad**— del `R²` del modelo desplegado (0,847) y del efecto de `Overall Qual` (+10,9 % por punto de calidad).
2. **¿Se cumplen los supuestos?** Una **tabla verificada** que ejecuta Breusch-Pagan, Durbin-Watson, VIF y Jarque-Bera con su umbral, veredicto y consecuencia. El **gráfico** de normalidad es el Q-Q del diagnóstico de supuestos (más arriba en esta Sección 8).

**🔎 Qué hace este código.** Lee del Excel las hojas `bootstrap_ic` (IC 95 % por remuestreo de 2000 casos, semilla 42) y `supuestos_panel` (Breusch-Pagan, Durbin-Watson, VIF y Jarque-Bera del modelo log). Muestra ambas y **verifica** que el IC bootstrap del efecto de `Overall Qual` **coincide** con el IC analítico robusto HC3 de la hoja `inferencia_robusta` —dos rutas independientes a la misma incertidumbre—: un `assert` cruza las dos vías sin volver a ajustar el modelo.

In [ ]:
# Panel de incertidumbre: leer del Excel (bootstrap_ic + supuestos_panel) y cruzar bootstrap vs HC3
from openpyxl import load_workbook
_wbp = load_workbook(RUTA_XLSX, data_only=True)

_boot = pd.DataFrame(list(_wbp["bootstrap_ic"].iter_rows(min_row=2, values_only=True)),
                     columns=[c.value for c in _wbp["bootstrap_ic"][1]])
_panel = pd.DataFrame(list(_wbp["supuestos_panel"].iter_rows(min_row=2, values_only=True)),
                      columns=[c.value for c in _wbp["supuestos_panel"][1]])

print("IC bootstrap 95 % de la cifra insignia (hoja bootstrap_ic; remuestreo de casos, no asume normalidad):")
print(_boot[["cifra", "escala", "punto", "ic95_bajo", "ic95_alto"]].to_string(index=False))
print("\nTabla de supuestos VERIFICADA (hoja supuestos_panel):")
print(_panel.to_string(index=False))

# Verificacion de DOS RUTAS: IC bootstrap del efecto de Overall Qual ~ IC analitico HC3
_inf = {r[1]: r for r in _wbp["inferencia_robusta"].iter_rows(min_row=2, values_only=True)
        if r[0] == "log_multiple"}
_oq = _inf["Overall Qual"]                         # (modelo, termino, coef, se, t, p, ic_bajo, ic_alto)
_hc3_eff_lo = (np.exp(_oq[6]) - 1) * 100
_hc3_eff_hi = (np.exp(_oq[7]) - 1) * 100
_bo = _boot.loc[_boot["cifra"] == "efecto_overall_qual_pct"].iloc[0]
print("\nDos rutas a la incertidumbre del efecto de Overall Qual (por punto de calidad):")
print("  IC bootstrap (percentil) : [%.2f %%, %.2f %%]" % (_bo["ic95_bajo"], _bo["ic95_alto"]))
print("  IC analitico HC3         : [%.2f %%, %.2f %%]" % (_hc3_eff_lo, _hc3_eff_hi))
assert abs(_bo["ic95_bajo"] - _hc3_eff_lo) < 0.5 and abs(_bo["ic95_alto"] - _hc3_eff_hi) < 0.5
print("OK: el IC bootstrap y el IC robusto HC3 coinciden (mas-menos 0,5 pp) — la incertidumbre no depende de la via.")

**📖 Cómo se lee.** El `R²` del modelo desplegado (0,847) tiene un **IC bootstrap del 95 %** de ~[0,831, 0,862]: aun con la incertidumbre del muestreo, el modelo explica **más del 83 %** de la varianza de `log(precio)`. El efecto de `Overall Qual` (+10,9 % por punto de calidad) cae con 95 % de confianza en ~[9,9 %, 12,0 %], y ese intervalo **coincide** con el de la vía analítica robusta HC3: dos métodos distintos —remuestreo no paramétrico y matriz sándwich— entregan la misma incertidumbre, de modo que la conclusión de negocio **no depende del supuesto de normalidad**. La **tabla de supuestos** deja la lectura honesta: Breusch-Pagan y Jarque-Bera **rechazan** (hay heterocedasticidad y no-normalidad), pero con n ≈ 2 923 el **teorema central del límite** protege la inferencia y los **IC robustos/bootstrap** ya no exigen normalidad; el VIF máximo (2,47) no muestra problema y el Durbin-Watson (~1,68) aparece rotulado como **«señal de dependencia espacial»**: el CSV viene agrupado geográficamente por barrio, así que el panel remite a los **SE agrupados por barrio** (hoja `se_cluster_barrio`), que ensanchan la incertidumbre honesta más que HC3. **Matiz sobre las «dos rutas»:** el bootstrap i.i.d. y el HC3 coinciden porque **comparten** el supuesto de independencia entre casas; frente a la dependencia intra-barrio ambos son optimistas, y el IC agrupado por barrio es el conservador (efecto de `Overall Qual` en [9,3 %; 12,5 %] frente a [9,9 %; 11,9 %]). Conclusión: el modelo es apto para tasar **siempre que su inferencia se reporte de forma robusta** (HC3 / bootstrap como piso; SE por barrio como lectura honesta de la dependencia espacial), no con los errores clásicos.

### La dummy leída en el modelo log — demo ejecutable de la P7 — capítulo 4.2 (subsección 8.10)

**🔎 Qué hace este código.** Ajusta el modelo `log(SalePrice) ~ Gr Liv Area + Garage Cars + Total Bsmt SF + premium` (n = 2923) y muestra la mecánica que evalúa la **P7 del control corto**: con la respuesta en log, el coeficiente de una **dummy** se lee como cambio porcentual **frente a la categoría de referencia** (la base de `drop_first`), con la forma exacta 100·(e^β − 1). Aquí β_premium ≈ **0,1535** ⇒ e^β − 1 ≈ **+16,6 %**: a igual superficie, garaje y sótano, una casa de barrio premium se vende ≈ 16,6 % más cara que su gemela de barrio estándar —una **diferencia contra la base**, no un nivel—. Las cifras viven en la hoja **`dummy_log`** del Excel (escrita por la celda del contrato) y las certifica el material de referencia de la sesión; el `assert` final verifica que la hoja coincide con el modelo recién ajustado.

In [ ]:
# Demo P7 - la dummy premium en el modelo log: beta ~ 0.15 -> e^beta - 1 ~ +16 %
X_dlog = sm.add_constant(lab[["Gr Liv Area", "Garage Cars", "Total Bsmt SF", "premium"]])
m_dlog = sm.OLS(np.log(lab["SalePrice"]), X_dlog).fit()
b_prem = float(m_dlog.params["premium"])
efecto_prem = (np.exp(b_prem) - 1) * 100

print("Modelo: log(SalePrice) ~ Gr Liv Area + Garage Cars + Total Bsmt SF + premium")
print("n                  = %d" % int(m_dlog.nobs))
print("beta_premium       = %.4f  (p = %.2e)" % (b_prem, m_dlog.pvalues["premium"]))
print("Aproximacion 100*b = %+.1f %%  (solo vale para beta pequeno)" % (100 * b_prem))
print("Lectura exacta     = e^beta - 1 = %+.1f %% frente a la categoria base (no premium)"
      % efecto_prem)

# Verificacion contra la hoja dummy_log del Excel (la escribe la celda del contrato)
from openpyxl import load_workbook
_wb_chk = load_workbook(RUTA_XLSX, data_only=True)
_dl = {r[0]: r[1] for r in _wb_chk["dummy_log"].iter_rows(min_row=2, values_only=True)}
assert abs(float(_dl["beta_premium_log"]) - b_prem) < 1e-12, "dummy_log: beta no coincide"
assert abs(float(_dl["efecto_pct_premium"]) - efecto_prem) < 1e-9, "dummy_log: efecto no coincide"
print()
print("assert OK: la hoja 'dummy_log' del Excel coincide con el modelo reajustado.")

## Práctica — Drills (ejercicios) (Sección 9 del cuaderno)

Enunciados de práctica; el detalle y la rúbrica están en `evaluacion/drills.docx` (las soluciones se entregan por separado).

**Drill 1 — VIF de tres predictores correlacionados.** Ajustar `SalePrice ~ Gr Liv Area + Garage Cars + Garage Area` y calcular el VIF de cada predictor. Identificar cuáles superan el umbral 5, explicar *por qué* (qué variables miden lo mismo) y proponer qué predictor eliminar o combinar. Interpretar el efecto de la colinealidad sobre los errores estándar.

**Drill 2 — Construir e interpretar una interacción zona × tamaño.** Elegir un barrio (o grupo de barrios) como «premium», crear la dummy y la interacción `Gr Liv Area × premium`, ajustar el modelo y **leer el coeficiente de la interacción**: ¿cuánto más (o menos) vale el pie² en la zona premium? Redactar la conclusión en unidades de negocio (USD/pie²), respetando el principio de jerarquía.

**Drill 3 — Comparar dos modelos por R² ajustado vs. RMSE en CV 5-fold.** Comparar `SalePrice ~ Gr Liv Area` contra `SalePrice ~ Gr Liv Area + Overall Qual + Year Built` usando (a) R² ajustado in-sample y (b) RMSE medio ± sd en CV 5-fold (`random_state=42`). Indicar qué modelo se elige y por qué, y qué se hace si ambos criterios discrepan.

## 4.9 — ¿Qué no se puede afirmar, y qué sigue en S05? Cierre (Sección 10 del cuaderno)

### Entregable evaluable
Ajustar un modelo de **regresión múltiple de precios de vivienda (*Ames*)** con **al menos una dummy y una interacción**, diagnosticar los supuestos (VIF, residuales, heterocedasticidad con Breusch-Pagan → robustos si aplica), **elegir el modelo por CV 5-fold** y comunicar los coeficientes parciales en unidades de negocio. Rúbrica vigesimal 0–20 en `evaluacion/entregable.docx`. Modalidad: parejas; entrega individual. Guía paso a paso: `laboratorio/GUIA_LABORATORIO_S04.docx`; plantillas de apoyo: `plantillas/checklist_diagnostico_regresion.docx`, `plantillas/comparacion_modelos_cv.docx`, `plantillas/diccionario_ames.docx`.

### Control corto
Habrá un control corto de la sesión (banco de ítems `[S04]`) sobre VIF, lectura de coeficientes parciales, interpretación de interacciones/dummies y elección de modelo por CV.

### Proyecto integrador
Esta sesión alimenta la fase de **Modelado** del proyecto integrador transversal: el modelo múltiple diagnosticado y validado por CV es la base sobre la que se construirán las técnicas de las sesiones siguientes.

### Materiales de apoyo de la sesión
- Guía del laboratorio: `laboratorio/GUIA_LABORATORIO_S04.docx`
- Plantillas: `plantillas/checklist_diagnostico_regresion.docx`, `plantillas/comparacion_modelos_cv.docx`, `plantillas/diccionario_ames.docx`
- Drills y entregable: `evaluacion/drills.docx`, `evaluacion/entregable.docx`
- Mapa de celdas del cuaderno: el cuaderno de la sesión
- Supuestos de la sesión (fuente canónica): la guía de supuestos de la sesión (ver «Sección 8»)
- Validación de la réplica (recomputa desde la base): el material de referencia de la sesión

### Para seguir explorando (actualidad — ver las fuentes de actualidad de la sesión)
Casos y regulación recientes de *pricing* hedónico y valoración automática (AVM):

1. **Tapia et al. (2025), PLOS ONE** — AVM (machine learning) vs. hedónica con ajustes espaciales en Santiago de Chile: el ML predice mejor, pero sus explicaciones (SHAP) se alinean con el OLS, recordando el cuidado al interpretar coeficientes con autocorrelación espacial. https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0318701
2. **CFPB et al. (vigente 01/10/2025)** — *Quality Control Standards for AVM*: la validación, el diagnóstico y la **no discriminación** de un modelo de precios pasan a ser exigencia regulatoria. https://www.consumerfinance.gov/rules-policy/final-rules/quality-control-standards-for-automated-valuation-models/
3. **Gorjian (2025), MPRA 125676** — revisión de 23 estudios: la hedónica conserva **poder explicativo** de los *drivers* de precio mientras el ML aporta precisión; lo óptimo es combinarlas. https://mpra.ub.uni-muenchen.de/id/eprint/125676
4. **Gümmer et al. (2025), arXiv 2508.03156** — regresión por segmento de mercado (43 000 propiedades alemanas): ilustra en producción la idea de que «el valor del metro cuadrado depende de la zona» (interacción zona × tamaño). https://arxiv.org/abs/2508.03156

### Lecturas base
- **ISLR** cap. 3.2–3.3 y 5.1 (statlearning.com) — texto principal.
- **De Cock, D. (2011)** — JSE 19(3); paper de la réplica.
- **White (1980)** (errores robustos); **Cook (1977)** (distancia de Cook); **Belsley, Kuh & Welsch (1980)** (colinealidad e influyentes).

Bibliografía completa verificada en la bibliografía de la sesión.

> **Alcance.** Esta sesión trabaja **regresión múltiple con diagnóstico, dummies e interacciones** y la elección de modelos por CV. La regularización y el modelado no lineal son S05; el PCA, S06; las series (autocorrelación como objeto), S11; la inferencia causal, S12. Aquí solo se nombran.